# MUSIC DATA PREPARATION PIPELINE (PYTHON ONLY) - FULL DEVELOPMENT VERSION
## PROJECT: Artist Page Lookup
---

- **This is the original, full development version of the project.** A shortened and polished version is available here: [Music Data Preparation Pipeline - Polished Version](https://github.com/ed-cybros/Music-Data-Preparation-Pipeline-Python-Only---Polished-Version).

- The raw version intentionally preserves intermediate approaches, corrections, repeated reviews, and development decisions that occurred during the project. **It is intended to document the reasoning and problem-solving process** rather than present only the final implementation.

## 1. PROJECT OVERVIEW

- This project is **an early attempt at building a complete data preparation pipeline** **applied specifically to music data**.

- It is **deliberately implemented in pure Python** to practice core data cleaning, transformation, validation, and data-structuring techniques without relying on specialized data-processing libraries.

- **The project features a modular, multi-pass data cleaning pipeline** featuring dedicated, single-column validation checks. This facilitates:
    - Easier debugging
    - Better readability (+ easier code audit and better quality control)
    - Explicit business logic
    <br>

- The pipeline takes a synthetically corrupted music dataset, inspects its quality, cleans and transforms the relevant data, validates the resulting structure and calculations, and **exports a simplified Artist Page lookup** dataset in JSON format.

- **The resulting Artist Page structure contains** aggregated information for each artist:

    - Average track popularity
    - Average BPM
    - Most frequently occurring genre
    <br>

- The project also **considers characteristics specific to music data** when deciding **how values should be interpreted and preserved**.

- The source dataset is the [Spotify - All Time Top 2000s Mega Dataset](https://www.kaggle.com/datasets/iamsumat/spotify-top-2000s-mega-dataset?resource=download), used here in a synthetically corrupted form for data-cleaning practice.

---

## 2. PROJECT PURPOSE

- The fundamental purpose of this project is **practical learning through implementation**.

- Rather than practicing individual cleaning or transformation techniques in isolation, **the goal was to work through a complete process** in which different decisions affect later stages of the pipeline.

- The project focuses on:

    - **Inspecting and understanding an unfamiliar dataset**
    - **Applying common data-cleaning and transformation procedures**
    - **Practicing data inspection, validation, transformation, and structuring** as parts of one coherent pipeline
    - **Structuring the resulting data** around a specific objective
    - **Preserving useful information** where possible
    - **Considering domain-specific characteristics** when making data-cleaning decisions
    <br>

- The raw version also **serves as documentation of the reasoning and problem-solving process** behind these decisions, both for private analysis and for any potential external third-party assesment or review.

---

## 3. PROJECT PIPELINE
- The project follows the following general process:

    - **Define the data objective**. Determine the information required for the final output.
    - **Load and inspect the dataset.** Examine the available columns, data types, missing values, duplicates, and other potential quality issues.
    - **Clean and transform the data.** Address identified issues and prepare the information required for the final structure.
    - **Validate the resulting data.** Check whether transformations produced the intended results and whether relevant information was preserved.
    - **Structure and export the data.** Aggregate track-level information into an artist-level lookup structure and export it as JSON.
    <br>

- The individual stages are not entirely independent. Inspection and validation are used throughout the process, particularly when a transformation introduces unexpected results or reveals additional characteristics of the data.

---

### STAGE 1 - DEFINING DATA OBJECTIVE

- **The goal is to transform track-level information into a simplified artist-level lookup structure.**

- For each artist, **the final dataset should contain**:

    - **The average popularity** of available tracks
    - **The average BPM** of available tracks
    - **The most frequently occurring genr**e
    <br>

- **The objective provides a concrete direction** for deciding which information is relevant, how it should be transformed, and which data-quality issues require attention.

- The resulting structure is intentionally simple. The focus of the project is not the complexity of the final output, but the process required to prepare the underlying data reliably.

---

### STAGE 2 - DATA LOAD & INSPECTION

- Before any data cleaning/transformation can be performed, the dataset must be thoroughly examined and its quality assessed.

- The dataset **will be inspected for**:
    - missing records,
    - invalid numerical values,
    - sentinel values,
    - inadequate or inconsistent values,
    - naming inconsistencies.

#### 1) --DATA LOAD--

- Safe data ingestion using the **csv reader module**.

- **Column names** will be extracted from the header row and displayed together with their corresponding indices **for quick reference** during the data exploration and preparation process.

- **Total row number** will be displayed for reference.

In [1]:
import csv

with open("messy_spotify_2000.csv", encoding="utf-8") as file:
    reader = list(csv.reader(file))
    header =  reader[0]
    rows = reader[1:]

    for i, h in enumerate(header):
        print(i, h)

    print("\nRows total:", len(rows))

0 Index
1 Title
2 Artist
3 Top Genre
4 Year
5 Beats Per Minute (BPM)
6 Energy
7 Danceability
8 Loudness (dB)
9 Liveness
10 Valence
11 Length (Duration)
12 Acousticness
13 Speechiness
14 Popularity

Rows total: 2044


#### 2) --DUPLICATE ROWS CHECK--

- First, **duplicate rows** will be checked for, rechecked, and counted, as their presence would skew any downstream calculations.

- The **Artist-Title** relationship **will be used as a candidate-duplicate key**. But since they may not be a proof that full rows are duplicated, further investigation and validation checks will be performed before any other actions in their regard.

- Candidate-duplicate rows **ID's will be retrieved** and emplyed **to perform validation checks**.

- In case of proved duplication, **the duplicate rows ID's will be extracted** and **the duplicate rows will be entirely removed from the dataset** before any downstream inspection procedures implementation.

#### --Stage 1. Identifying Duplicate Rows

##### Artist-Title Candidate-Duplicate Key Implementation

In [2]:
unique_tups = set()
dupl_tups = set()
dupl_tups_count = 0
row_id_dic = {}  # Dictionary to store all rows IDs.

for i, row in enumerate(rows):
    artist = row[2]
    title = row[1]
    tup = (artist, title)  # Represent a unique row via Artist - Title unique relationship.
    
    row_id_dic[tup] = row_id_dic.get(tup, [])  # Store a list of indeces per tuple.
                                               # If a tuple has a duplicate, more than one index will be stored into the list.
    
    if tup not in unique_tups:
        unique_tups.add(tup)
        row_id_dic[tup].append(i)
    else:
        dupl_tups.add(tup)
        #print(row)
        dupl_tups_count += 1
        print("Duplicate:", tup in unique_tups)
        print("Total duplicate tuples:", dupl_tups_count)
        row_id_dic[tup].append(i)

intersection = dupl_tups.intersection(dupl_tups)

print("\nUnique tuples total:", len(unique_tups))
print("Duplicate tuples total:", len(dupl_tups))
print("Duplicate totals match:", dupl_tups_count == len(dupl_tups))
print("Duplicate values match:", intersection == dupl_tups)


Duplicate: True
Total duplicate tuples: 1
Duplicate: True
Total duplicate tuples: 2
Duplicate: True
Total duplicate tuples: 3
Duplicate: True
Total duplicate tuples: 4
Duplicate: True
Total duplicate tuples: 5
Duplicate: True
Total duplicate tuples: 6
Duplicate: True
Total duplicate tuples: 7
Duplicate: True
Total duplicate tuples: 8
Duplicate: True
Total duplicate tuples: 9
Duplicate: True
Total duplicate tuples: 10
Duplicate: True
Total duplicate tuples: 11
Duplicate: True
Total duplicate tuples: 12
Duplicate: True
Total duplicate tuples: 13
Duplicate: True
Total duplicate tuples: 14
Duplicate: True
Total duplicate tuples: 15
Duplicate: True
Total duplicate tuples: 16
Duplicate: True
Total duplicate tuples: 17
Duplicate: True
Total duplicate tuples: 18
Duplicate: True
Total duplicate tuples: 19
Duplicate: True
Total duplicate tuples: 20
Duplicate: True
Total duplicate tuples: 21
Duplicate: True
Total duplicate tuples: 22
Duplicate: True
Total duplicate tuples: 23
Duplicate: True
Tota

##### Candidate-Duplicate Rows ID Extraction

In [3]:
# Scan through the dictionary to reveal duplicate tuples ID's.

i = 0
dupl_id_list = []  # Create a list to store

for k, v in row_id_dic.items():
    if len(v) > 1:  # Duplicate tuples have more than one index in the list.
        i +=1
        id_tup = tuple(v)
        dupl_id_list.append(id_tup)
        print(i, id_tup)

1 (5, 1998)
2 (21, 2020)
3 (38, 2042)
4 (56, 2032)
5 (104, 2011)
6 (123, 2025)
7 (128, 2043)
8 (177, 2009)
9 (207, 2010)
10 (260, 2037)
11 (287, 2013)
12 (291, 2000)
13 (297, 2034)
14 (324, 2019)
15 (362, 2007)
16 (521, 2040)
17 (536, 1996)
18 (558, 1995)
19 (598, 2033)
20 (792, 2004)
21 (833, 2012)
22 (922, 2028)
23 (1052, 2029)
24 (1160, 2024)
25 (1170, 2003)
26 (1217, 2036)
27 (1248, 2027)
28 (1268, 2021)
29 (1285, 2035)
30 (1288, 2001)
31 (1314, 2002)
32 (1338, 2014)
33 (1349, 2005)
34 (1387, 2016)
35 (1388, 2017)
36 (1397, 2015)
37 (1407, 2022)
38 (1490, 2031)
39 (1518, 2030)
40 (1552, 2023)
41 (1570, 1997)
42 (1674, 2018)
43 (1676, 2006)
44 (1683, 2026)
45 (1744, 1999)
46 (1766, 2008)
47 (1857, 2041)
48 (1867, 2039)
49 (1878, 1994)
50 (1973, 2038)


##### Candidate-Duplicate Rows Duplication Validation

In [4]:
# Use the extracted ID's to confirm duplication of the full rows by comparing the pairs directly.

identical_count = 0
unidentical_count = 0

row_pairs_dic = {}  # Store the pairs of candidate-duplicate rows in a dictionary in case of any further manipulations need. 

for j, k in dupl_id_list:
    if rows[j] == rows[k] in enumerate(rows):
        identical_count += 1
        print(identical_count, "- Identical rows")
        
    else:
        unidentical_count += 1
        print(unidentical_count, "- Rows are NOT identical")
        row_a = rows[j]
        row_b = rows[k]
        print(row_a)
        print(row_b)
        print()
    

1 - Rows are NOT identical
['6', '  The Road Ahead (Miles Of The Unknown)  ', 'City To City', 'alternative pop rock', '2004', '99', '46', '54', '-9', '14', '14', 'null', '0', '2', '45']
['6', '  The Road Ahead (Miles Of The Unknown)  ', 'City To City', 'alternative pop rock', '2004', '99', '46', '54', '-9', '14', '14', 'null', '0', '2', '999']

2 - Rows are NOT identical
['22', 'The Cave', 'Mumford & Sons', 'modern folk rock', '2009', '142', '51', '60', '-10', '11', '35', '218', '5', '4', '67']
['22', 'The Cave', 'Mumford & Sons', 'modern folk rock', '2009', '142', '51', '60', '-10', '11', '35', '218', '5', '4', '999']

3 - Rows are NOT identical
['39', 'All My Life', 'Foo Fighters', 'alternative metal', '2002', '168', '60', '58', '-6', '48', '65', '263', '0', '5', '72']
['39', 'All My Life', 'Foo Fighters', 'alternative metal', '2002', '168', '60', '58', '-6', '48', '65', '263', '0', '5', '999']

4 - Rows are NOT identical
['57', "Sometimes You Can't Make It On Your Own", 'U2', 'irish

##### **CONCLUSIONS DEDUPLICATION STAGE 1:**
---
- **50 pairs of candidate-duplicate rows** identified (i.e. **pairs of rows with identical Artist-Title relationship**).

- **0 fully identical rows** identified. Which implies some of the data beyond the Artist-Title relationship differs between the rows.

- **Deeper investigation** of candidate-duplicate rows is required to identify the difference in values within each pair for decision-making about which row should be retained and which row should be removed.

- Next, a **value-by-value comparison** will be performed for each pair.
---

#### --Stage 2. Deeper Candidate-Duplicate Rows Investigation

##### Value-by-Value Comparison

In [5]:
# Modify the above code with value-by-value comparison.

dupl_val_col = set()  # Identify and store all columns where differences between values occur.

for j, k in dupl_id_list:
    if rows[j] == rows[k] in enumerate(rows):
        continue
        
    else:
        row_a = rows[j]
        row_b = rows[k]

        # Add value-by-value comparison
        
        for i, val in enumerate(zip(row_a, row_b)):
            if val[0] != val[1]:
                dupl_val_col.add(header[i])
                print(header[i], val[0], val[1])

print("Columns:", dupl_val_col)


Popularity 45 999
Popularity 67 999
Popularity 72 999
Popularity 57 999
Popularity 58 999
Popularity 70 999
Popularity 42 999
Popularity 47 999
Popularity 72 999
Popularity 43 999
Popularity  999
Popularity 62 999
Popularity  999
Popularity 999999 999
Popularity 36 999
Popularity 57 999
Popularity  999
Popularity 999999 999
Popularity 49 999
Popularity 56 999
Popularity 58 999
Popularity 65 999
Popularity 56 999
Popularity  999
Popularity 62 999
Popularity 68 999
Popularity 999999 999
Popularity 67 999
Popularity 76 999
Popularity  999
Popularity UNKNOWN 999
Popularity UNKNOWN 999
Popularity 53 999
Popularity 73 999
Popularity 80 999
Popularity 76 999
Popularity 62 999
Popularity 60 999
Popularity 69 999
Popularity 999999 999
Popularity 74 999
Popularity 66 999
Popularity 49 999
Popularity 72 999
Popularity 51 999
Popularity  999
Popularity 74 999
Popularity 57 999
Popularity 999999 999
Popularity  999
Columns: {'Popularity'}


##### **CONCLUSIONS DEDUPLICATION STAGE 2:**
---
- The deeper inspection reveals that **only one column (Popularity) features differences** within any given row pair.

- Quick observation of the displayed values reveals **numerous cases of out-of-bounds sentinel values featured by one or both of the rows** within the pairs.

- **Further inspection will be done** to investigate and compare **the Popularity values specifically** within each row pair **to make a decision about which row within a pair can be safely removed**.

- Next, a **filter** will be applied **to identify non-numeric** (popularity values should be numeric) **and out-of-bounds** (the dataset features popularity score between 0 and 100) values. The logic behind which row to keep and which row to delete is as follows:

    - **If one of the rows** within a pair contains a proper value, while the other contains an improper one, the row with the proper value will be retained, the other row will be removed.
      
    - **If both rows** within a pair contain **improper** values, this means that any of them can be kept and the other removed. The first row within a pair will be retained by default.
 
    - **If both rows** within a pair contain **proper** values, **a manual inspection and decision-making** will be required for each individual case.
---

#### --Stage 3. Decision-Making

##### Value Filtering

In [6]:
# Modify the above code again with adding a filter to rule out one of the rows within each pair.

count = 0
drop_row_id_list = []  # Store ID's of the rows that are to be deleted.
disput_case_count = 0  # Supervise and count disputable cases that require deliberate decision-making

for j, k in dupl_id_list:
    count += 1
    print("ROW PAIR #", count)
    print("- - - - - - -")
    if rows[j] == rows[k] in enumerate(rows):
        continue
        
    else:
        row_a = rows[j]
        row_b = rows[k]
        for i, val in enumerate(zip(row_a, row_b)):
            val_a = val[0]
            val_b = val[1]
            
            if val_a != val_b:
                dupl_val_col.add(header[i])

                # Add filter ruling out non-numeric or out-of-bounds values:

                if val_a.isdigit() and (0 < int(val_a) <= 100):
                    pass_val_a = True
                    print(f"{pass_val_a}\n|__VAL A: {val_a}\n|__ROW INDEX: {j}")
                else:
                    pass_val_a = False
                    print(f"{pass_val_a}\n|__VAL A: {val_a}\n|__ROW INDEX: {j}")
                if val_b.isdigit() and (0 < int(val_b) <= 100):
                    pass_val_b = True
                    print(f"{pass_val_b}\n|__VAL A: {val_b}\n|__ROW INDEX: {k}")
                else:
                    pass_val_b = False
                    print(f"{pass_val_b}\n|__VAL B: {val_b}\n|__ROW INDEX: {k}")

                if pass_val_a and not pass_val_b:
                    drop_row_id_list.append(k)
                    print(f"Index B appended to drop list: {k}")

                elif pass_val_b and not pass_val_a:
                    drop_row_id_list.append(j)
                    print(f"Index A appended to drop list: {j}")

                elif not pass_val_a and not pass_val_a:
                    drop_row_id_list.append(k)
                    print()
                    print(f"Value A: {val_a}\nValue B: {val_b}")
                    print(f"Index B appended to drop list: {k}")
                    print("_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _")

                elif pass_val_a and pass_val_a:
                    disput_case_count += 1
                    print()
                    print(f"Value A: {val_a}\nValue B: {val_b}")
                    print("Disputable case. Manual inspection required")
                    print("* * * * * * * * * * * * * * * * * * *")

                print()

print(f"Total disputable cases: {disput_case_count}")
print(f"Total ID's retreived: {len(drop_row_id_list)}")
print(drop_row_id_list)

ROW PAIR # 1
- - - - - - -
True
|__VAL A: 45
|__ROW INDEX: 5
False
|__VAL B: 999
|__ROW INDEX: 1998
Index B appended to drop list: 1998

ROW PAIR # 2
- - - - - - -
True
|__VAL A: 67
|__ROW INDEX: 21
False
|__VAL B: 999
|__ROW INDEX: 2020
Index B appended to drop list: 2020

ROW PAIR # 3
- - - - - - -
True
|__VAL A: 72
|__ROW INDEX: 38
False
|__VAL B: 999
|__ROW INDEX: 2042
Index B appended to drop list: 2042

ROW PAIR # 4
- - - - - - -
True
|__VAL A: 57
|__ROW INDEX: 56
False
|__VAL B: 999
|__ROW INDEX: 2032
Index B appended to drop list: 2032

ROW PAIR # 5
- - - - - - -
True
|__VAL A: 58
|__ROW INDEX: 104
False
|__VAL B: 999
|__ROW INDEX: 2011
Index B appended to drop list: 2011

ROW PAIR # 6
- - - - - - -
True
|__VAL A: 70
|__ROW INDEX: 123
False
|__VAL B: 999
|__ROW INDEX: 2025
Index B appended to drop list: 2025

ROW PAIR # 7
- - - - - - -
True
|__VAL A: 42
|__ROW INDEX: 128
False
|__VAL B: 999
|__ROW INDEX: 2043
Index B appended to drop list: 2043

ROW PAIR # 8
- - - - - - -
True


##### **CONCLUSIONS DEDUPLICATION STAGE 3:**
---
- 50 condidate-duplicate row pairs have been processed.

- 50 rows have been filtered out for complete removal from the dataset and their ID's retreived.

- 0 instances of disputable cases where both rows within a pair contain proper values.

- Next, **the list of retrieved ID's will be used to remove the ruled out rows from the dataset**.
---

#### --Stage 4. Duplicate Rows Removal & Results Validation

##### Rows Removal

In [7]:
removed_rows = []  # A list to set all the rows removed from the dataset for further validation.
drop_row_id_list.sort(reverse=True)

for i in drop_row_id_list:
    print(rows[i])
    print(i)
    x = rows.pop(i)
    print(x)
    removed_rows.append(x)
    

['129', 'De wedstrijd', 'Bram Vermeulen', 'belgian rock', '2006', '136', '85', '58', '-6', '79', '58', '302', '53', '5', '999']
2043
['129', 'De wedstrijd', 'Bram Vermeulen', 'belgian rock', '2006', '136', '85', '58', '-6', '79', '58', '302', '53', '5', '999']
['39', 'All My Life', 'Foo Fighters', 'alternative metal', '2002', '168', '60', '58', '-6', '48', '65', '263', '0', '5', '999']
2042
['39', 'All My Life', 'Foo Fighters', 'alternative metal', '2002', '168', '60', '58', '-6', '48', '65', '263', '0', '5', '999']
['1858', 'In My Life - Remastered 2009', 'Beatles', 'british invasion', '1965', '103', '44', '69', '-11', '11', '44', '146', '45', '3', '999']
2041
['1858', 'In My Life - Remastered 2009', 'Beatles', 'british invasion', '1965', '103', '44', '69', '-11', '11', '44', '146', '45', '3', '999']
['522', 'Lola Montez', 'Volbeat', 'alternative metal', '2013', '152', '88', '51', '-5', '7', '68', '268', '0', '3', '999']
2040
['522', 'Lola Montez', 'Volbeat', 'alternative metal', '201

##### Removal Results Validation

In [8]:
# Use the list of removed rows, extract Artist-Title relationships, convert into a tuple and store into a set.
removed_tups = set()

for x in removed_rows:
    x = (x[2], x[1])
    removed_tups.add(x)

# Display length of each set
print(len(removed_tups))
print(len(dupl_tups))

# Use set issubset and difference methods to validate the complete match
print(removed_tups.issubset(dupl_tups))
print(len(removed_tups.difference(dupl_tups)))

print("Deduplicated dataset rows total:", len(rows))
        

50
50
True
0
Deduplicated dataset rows total: 1994


##### **CONCLUSIONS DEDUPLICATION STAGE 4 (FINAL)**
---
- **The validation check confirmed correct row removal** from the dataset.

- **50 rows have been removed** from the dataset.

- **1994 rows remain** in the dataset after deduplication.

- The **dataset is ready for further inspection**.


---
---

#### 3) --GENERAL MISSING DATA COUNT & INSPECTION--

-  An approximate **total number of rows with incomplete records** will be counted bfiefly within the targeted subset of the data, **to assess overall subset data integrity**.

-  Any registered incomplete record will be **displayed** along with the full ROW it belongs to and the corresponding COLUMN name **for examination and validation**.

- The total number of **incomplete records per column** will be also counted and displayed.

- A simple **percentage calculating function** will be introduced **to quickly calculate and display what fraction of the total data** (dataset rows) a given number of missing entries make.

##### Percentage Calculating Function

In [9]:
def calc_percent(value):
    x = (value * 100) / len(rows)
    return f"({round(x, 2)}%)"

##### General Missing Data Count & Inspection

In [10]:
nan_list = ['unknown', 'null', 'n/a', 'none', '']
incomplete_rows = 0
col_dic = {}

for row in rows:
    incomplete_row = False
    for col_name, value in zip(header, row):
        if col_name in header[2:4] or col_name in header[5] or col_name in header[14]:
            try:
                value  = value.lower()
                if value in nan_list:
                    col_dic[col_name] = col_dic.get(col_name, 0) + 1
                    print(f"\n< > Missing record:\n|__ROW:{row}\n|__COLUMN: {col_name}\n|__VALUE: {value}")
                    incomplete_row = True
            except ValueError:
                print(f"\n   <!> ValueError occurred:\n  ||__ROW:{row}\n|__COLUMN: {col_name}\n  ||__VALUE: {value}")
                continue
        elif col_name in header[4:6] or col_name in header[14]:
            try:
                value  = int(value)
                if value == 0:
                    col_dic[col_name] = col_dic.get(col_name, 0) + 1
                    print(f"\n< >   'Zero' value:\n  ||__ROW:{row}\n  ||__COLUMN: {col_name}\n|__VALUE: {value}")
                    incomplete_row = True
            except ValueError:
                print(f"\n   <!> Conversion failed. ValueError occurred:\n  ||__ROW:{row}\n  ||__COLUMN: {col_name}\n  ||__VALUE: {value}")
                continue
            
    if incomplete_row:
        incomplete_rows += 1
        print("\n",col_dic)
        print("Total incomplete rows:", incomplete_rows)

print()
for k, v in col_dic.items():
    print(f"{k}: {v} {calc_percent(v)}")
print("Total incomplete rows:", incomplete_rows, calc_percent(incomplete_rows))


< > Missing record:
|__ROW:['1', 'Sunrise', 'Norah Jones', 'adult standards', '2004', '157', '30', '53', '-14', '11', '68', '201', '94', '3', 'UNKNOWN']
|__COLUMN: Popularity
|__VALUE: unknown

 {'Popularity': 1}
Total incomplete rows: 1

< > Missing record:
|__ROW:['12', 'Seven Nation Army', 'The White Stripes', 'alternative rock', '2003', '124', '46', '74', '-8', '26', '32', '232', '1', '8', '']
|__COLUMN: Popularity
|__VALUE: 

 {'Popularity': 2}
Total incomplete rows: 2

< > Missing record:
|__ROW:['14', "I'm going home", 'Ten Years After', 'album rock', '2005', '117', '93', '38', '-2', '81', '40', '639', '18', '10', 'UNKNOWN']
|__COLUMN: Popularity
|__VALUE: unknown

 {'Popularity': 3}
Total incomplete rows: 3

< > Missing record:
|__ROW:['20', 'Cry Me a River', 'Justin Timberlake', 'dance pop', '2002', '74', '65', '62', '-7', '10', '56', '288', '57', '18', '']
|__COLUMN: Popularity
|__VALUE: 

 {'Popularity': 4}
Total incomplete rows: 4

< > Missing record:
|__ROW:['37', 'Iris',

##### **CONCLUSIONS:**
---
- The initial row integrity inspection **reveals**:

    - 263 total rows with missing records (13.19% of total data),
    - 201 missing records in 'Popularity' column (10.08% of total data),
    - 73 missing records in 'Artist' column (3.66% of total data).
    - 0 missing records in the rest of the subset.
        
    
- Rows with missing records in **'Artist'** column, making **about 3.7%** of the total data, **are to be excluded** from the final dataset file, due to being the foundation value of the final project file and due to the inability to fill in the missing information.

- Rows with missing records in **'Popularity'** column, making **about 10.1%** of the total data, **are to be further inspected** and analized before any decision-making in their regard.

- Despite the fact of 0 missing records registered in the rest of the columns during the initial scan, **deeper inspection of the rest of the columns** is also to be performed.
---

In [11]:
artists_dic = {}
for row in rows:
    artist = row[2]
    artists_dic[artist] = artists_dic.get(artist, 0) +1
#print(artists_dic)
artists_list = list(artists_dic.items())

artists_list.sort(key=lambda x: x[1], reverse=True)
for i, v in enumerate(artists_list):
    print(i, v)

0 ('U2', 24)
1 ('The Rolling Stones', 24)
2 ('None', 23)
3 ('Bruce Springsteen', 22)
4 ('ABBA', 21)
5 ('David Bowie', 19)
6 ('', 18)
7 ('Fleetwood Mac', 18)
8 ('BLØF', 17)
9 ('UNKNOWN', 17)
10 ('Elvis Presley', 16)
11 ('nan', 16)
12 ('George Michael', 16)
13 ('NULL', 15)
14 ('Marco Borsato', 15)
15 ('Muse', 14)
16 ('The Beatles', 14)
17 ('Queene', 14)
18 ('Anouk', 13)
19 ('Queen', 13)
20 ('Dire Straits', 13)
21 ('De Dijk', 12)
22 ('Adele', 12)
23 ('Golden Earring', 12)
24 ('Bee Gees', 12)
25 ('Ed Sheeran', 12)
26 ('Creedence Clearwater Revival', 12)
27 ('Pink Floyd', 12)
28 ('Eagles', 12)
29 ('Andre Hazes', 11)
30 ('Beatles', 11)
31 ('The Beatels', 11)
32 ('Nirvana', 11)
33 ('Prince', 11)
34 ('coldplay ', 10)
35 ('Metallica', 10)
36 ('Boudewijn de Groot', 10)
37 ('Billy Joel', 10)
38 ('Coldplay', 10)
39 ('Michael Jackson', 10)
40 ('Elton John', 10)
41 ('Supertramp', 10)
42 ('Queen ', 10)
43 ('Foo Fighters', 9)
44 ('Johnny Cash', 9)
45 ('Phil Collins', 9)
46 ('Imagine Dragons', 9)
47 ('

#### 4) --ARTIST COLUMN INSPECTION--

- **Empty** and **unknown records** are to be looked for, counted and examined if present.

- **Multiple artists per track** are to be scanned for to check the formatting consistency.

##### Missing Records Scan

In [12]:
missing_artist_records = 0
missing_artist_row_id = []

for row in rows:
    row_id = row[0]
    try:
        artist = row[2].strip().lower()
        if artist in nan_list:
            missing_artist_records += 1
            missing_artist_row_id.append(row_id)
            print(f"\n< > Missing artist record:\n|__ROW:{row}\n|__VALUE: {artist}\n|__Total missing artist records: {missing_artist_records}")
    except ValueError:
        print(f"\n   <!> ValueError occurred:\n  ||__ROW:{row}\n  ||__VALUE: {artist}")
        continue

print("\nTotal missing artist records:", missing_artist_records, calc_percent(missing_artist_records))


< > Missing artist record:
|__ROW:['37', 'Iris', 'None', 'alternative rock', '2007', '156', '79', '29', '-6', '8', '51', '290', '0', '4', '75']
|__VALUE: none
|__Total missing artist records: 1

< > Missing artist record:
|__ROW:['68', 'In The Army Now', 'UNKNOWN', 'album rock', '2002', '105', '73', '68', '-8', '14', '94', '281', '11', '2', '']
|__VALUE: unknown
|__Total missing artist records: 2

< > Missing artist record:
|__ROW:['70', 'Maybe Tomorrow', 'NULL', 'britpop', '2003', '81', '65', '50', '-7', '33', '57', '273', '22', '4', 'UNKNOWN']
|__VALUE: null
|__Total missing artist records: 3

< > Missing artist record:
|__ROW:['71', 'Make You Feel My Love', '', 'british soul', '2008', '77', '18', '55', '-11', '11', '9', '212', '89', '3', '73']
|__VALUE: 
|__Total missing artist records: 4

< > Missing artist record:
|__ROW:['72', 'No One Knows', 'UNKNOWN', 'alternative metal', '2002', '171', '58', '51', '-5', '37', '67', '279', '3', '6', '65']
|__VALUE: unknown
|__Total missing art

##### Multiple Artists per Track Scan

In [13]:
mult_artists_dic = {}

mult_artists_sub = [";", "feat", " ft",  "&", "with", " vs. ", " x "]

for row in rows:
    artist = row[2].strip().lower()
    if any(substring in artist for substring in mult_artists_sub):
        mult_artists_dic[artist] = mult_artists_dic.get(artist, 0) + 1

for i, (k, v) in enumerate(mult_artists_dic.items()):
    print(i, k.title(), v)

0 Mumford & Sons 7
1 Veldhuis & Kemper 1
2 Within Temptation 3
3 Nick & Simon 2
4 Womack & Womack 1
5 Herman Brood & His Wild Romance 4
6 Ike & Tina Turner 2
7 Emerson, Lake & Palmer 2
8 Macklemore & Ryan Lewis 1
9 Derek & The Dominos 1
10 Cliff Richard & The Drifters 1
11 Earth, Wind & Fire 3
12 Suzan & Freek 2
13 Simon & Garfunkel 7
14 Crosby, Stills, Nash & Young 3
15 Earth & Fire 3
16 Bill Withers 2
17 Steve Harley & Cockney Rebel 1
18 Gladys Knight & The Pips 1
19 Bob Marley & The Wailers 7
20 The Mark & Clark Band 1
21 Jon & Vangelis 1
22 Elvis Costello & The Attractions 1
23 Nick Cave & The Bad Seeds 2
24 Gerry & The Pacemakers 1
25 The Mamas & The Papas 2
26 James Brown & The Famous Flames 1
27 Cuby & The Blizzards 3
28 Crosby, Stills & Nash 1


##### **CONCLUSIONS:**
---
- **73 missing artist records** registered (less, than 4% of the total data). Since Artist name is the cornerstone of the Artist Lookup project, the rows missing this vital information **will have to be dropped**. The ID's of rows with missing artist records were collected into a list to be used to skip the incomplete rows during the cleaning and export stage.


- **Semantic ambiguity cases** between artist entities found, due to the equal use of "&" both as:
    -  an intrinsic part of **a permanent artist name/group** (e.g. "Gladys Knight & The Pips", "Mumford & Sons")
    -  an indicator of a collaboration between independent artists (e.g. "Jon & Vangelis", "James Brown & The Famous Flames").

<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Such cases may need different approach depending on a given use case. For a simple Artist Page Lookup, which is potentially intended for human audience in the first place, collaborations will be kept as unique author entries, marking them as distinct artistic events. The best practice would be to keep both collaborations as unique single entities AND split them into individual authors. This will preserve all information and make the dataset suitable for a wider range of use cases.

---
---

##### **EXTRA: MISSING ARTIST-YEAR ASSOCIATION MINI ANALYSIS**
(This step could be skipped entirely, especially considering the fact that the original dataset has 0 missing artist records and the current dataset was broken synthetically, but, for the sake of pure practice and personal curiosity, the analysis will be performed nonetheless.)

- The Year values will be inspected together with missing artist records **to explore whether missing artist metadata is associated with a particular decade**.

- First, **two lists of Year values will be created**: one containing years from records with missing artist values, and another containing years from all records in the dataset.

- Second, the collected years will be **separated into decade bins** to calculate:

    - the total number of records per decade,
    - the number of records with missing artist values per decade.
    <br>
- Third, **the missing artist rate per decade will be calculated** by dividing missing artist records by the total number of records in each decade. These rates will **then be compared to analyze** whether missing artist records show any noticeable decade-based pattern.

##### Step 1 - Creating Year Lists

In [14]:
year_list_missing = []
year_list_all = []

for row in rows:
    year = row[4]
    artist = row[2]
    try:
        year = int(year)
        artist = row[2].lower()
        if artist in nan_list:
            year_list_missing.append(year)
    except ValueError:
        continue
    year_list_all.append(year)

year_list_missing.sort()
year_list_all.sort()
print(year_list_missing)

[1956, 1963, 1965, 1966, 1968, 1970, 1970, 1971, 1971, 1971, 1971, 1972, 1972, 1973, 1976, 1977, 1977, 1979, 1979, 1980, 1980, 1982, 1982, 1982, 1983, 1984, 1984, 1985, 1985, 1986, 1986, 1987, 1988, 1989, 1990, 1991, 1994, 1994, 1995, 1995, 1997, 1997, 1999, 2000, 2000, 2001, 2001, 2002, 2002, 2003, 2003, 2004, 2004, 2006, 2006, 2007, 2007, 2008, 2009, 2009, 2010, 2010, 2011, 2012, 2012, 2012, 2012, 2014, 2015, 2015, 2015, 2016, 2017]


##### Step 2 - Using a Dictionary to Distribute and Count Missing Artist Records per Decade

In [15]:
decade_dic_missing = {}  # Number of missing artists by decade
decade_dic_all = {}  # Number of all artist by decade

for year in year_list_missing:
    if 1950 <= year < 1960:
        decade_dic_missing["50s"] = decade_dic_missing.get("50s", 0) + 1
    elif 1960 <= year < 1970:
        decade_dic_missing["60s"] = decade_dic_missing.get("60s", 0) + 1
    elif 1970 <= year < 1980:
        decade_dic_missing["70s"] = decade_dic_missing.get("70s", 0) + 1
    elif 1980 <= year < 1990:
        decade_dic_missing["80s"] = decade_dic_missing.get("80s", 0) + 1
    elif 1990 <= year < 2000:
        decade_dic_missing["90s"] = decade_dic_missing.get("90s", 0) + 1
    elif 2000 <= year < 2010:
        decade_dic_missing["2000s"] = decade_dic_missing.get("2000s", 0) + 1  
    elif 2010 <= year < 2020:
        decade_dic_missing["2010s"] = decade_dic_missing.get("2010s", 0) + 1

for year in year_list_all:
    if 1950 <= year < 1960:
        decade_dic_all["50s"] = decade_dic_all.get("50s", 0) + 1
    elif 1960 <= year < 1970:
        decade_dic_all["60s"] = decade_dic_all.get("60s", 0) + 1
    elif 1970 <= year < 1980:
        decade_dic_all["70s"] = decade_dic_all.get("70s", 0) + 1
    elif 1980 <= year < 1990:
        decade_dic_all["80s"] = decade_dic_all.get("80s", 0) + 1
    elif 1990 <= year < 2000:
        decade_dic_all["90s"] = decade_dic_all.get("90s", 0) + 1
    elif 2000 <= year < 2010:
        decade_dic_all["2000s"] = decade_dic_all.get("2000s", 0) + 1  
    elif 2010 <= year < 2020:
        decade_dic_all["2010s"] = decade_dic_all.get("2010s", 0) + 1

print("Missing artists per decade:")
for decade, number in decade_dic_missing.items():
    print(decade, number)

print()

print("Total artists per decade:")
for decade, number in decade_dic_all.items():
    print(decade, number)


Missing artists per decade:
50s 1
60s 4
70s 14
80s 15
90s 9
2000s 17
2010s 13

Total artists per decade:
50s 9
60s 158
70s 353
80s 344
90s 331
2000s 400
2010s 399


##### Step 3 - Calculating Percentage per Decade

In [16]:
decades = decade_dic_all.keys()
percentage_dic = {}

for decade in decades:
    percentage_dic[decade] = round(decade_dic_missing[decade] / decade_dic_all[decade] * 100, 1)

print("Missing artist percentage per decade:")
for dec, perc in percentage_dic.items():
    print(f"{dec}: {perc}%")
    

Missing artist percentage per decade:
50s: 11.1%
60s: 2.5%
70s: 4.0%
80s: 4.4%
90s: 2.7%
2000s: 4.2%
2010s: 3.3%


##### **CONCLUSION:**
---
- **The proportion of missing artist records does not appear to be strongly associated with a particular decade.** After normalizing by the total number of records per decade, most decades show similar missing artist rates (approximately 2.5–4.2%). The 1950s show a higher missing rate (11.1%), but this is based on only 9 total records, making this estimate less reliable. Overall, there is no clear evidence of a decade-based pattern in missing artist records.
---

**NOTE:** 

Steps 2 and 3 could be performed **more quickly with pandas** library and its **.cut method**.

---

##### Step 2 (pandas) - Using pandas' .cut to Distribute and Count Missing Artist Records per Decade

In [17]:
import pandas as pd

# Create bins:
decades = [1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]


# Distributing missing records per decade:
for i in range(len(year_list_missing)):
    year_list_missing[i] = int(year_list_missing[i])
    
decade_categories_missing = pd.cut(year_list_missing, decades, right=False)


# Distributing all artist records per decade:
for i in range(len(year_list_all)):
    year_list_all[i] = int(year_list_all[i])
    
decade_categories_all = pd.cut(year_list_all, decades, right=False)

missing_counts = decade_categories_missing.value_counts()
total_coutns = decade_categories_all.value_counts()


print("Missing artists per decade:")
print(missing_counts,"\n")
print("Total artists per decade:")
print(total_coutns)


Missing artists per decade:
[1950, 1960)     1
[1960, 1970)     4
[1970, 1980)    14
[1980, 1990)    15
[1990, 2000)     9
[2000, 2010)    17
[2010, 2020)    13
Name: count, dtype: int64 

Total artists per decade:
[1950, 1960)      9
[1960, 1970)    158
[1970, 1980)    353
[1980, 1990)    344
[1990, 2000)    331
[2000, 2010)    400
[2010, 2020)    399
Name: count, dtype: int64


##### Step 3 (pandas) - Calculating Percentage per Decade

In [18]:
percentages =(missing_counts / total_coutns * 100).round(1)

print("Missing artist percentage per decade:")
print(percentages)

Missing artist percentage per decade:
[1950, 1960)    11.1
[1960, 1970)     2.5
[1970, 1980)     4.0
[1980, 1990)     4.4
[1990, 2000)     2.7
[2000, 2010)     4.2
[2010, 2020)     3.3
Name: count, dtype: float64


---

#### 5) --GENRE COLUMN INSPECTION--

- **Empty** and **unknown records** are to be looked for, counted and examined if present.

- **Unique genres** are to be collected into a set and displayed for inspection.

##### Missing Records Scan

In [19]:
missing_genre_records = 0

for i, row in enumerate(rows):
    genre = row[3].strip().lower()

    if genre in nan_list:
        missing_genre_records += 1
        print(f"\n< > Empty genre record:\n|__ROW:{row}\n|__Total missing genre records: {missing_genre_records}")

print("\nTotal missing genre records:", missing_genre_records, calc_percent(missing_genre_records))


Total missing genre records: 0 (0.0%)


##### Unique Genres Inspection

In [20]:
unique_genres = set()

for i, row in enumerate(rows):
    genre = row[3].strip().lower()

    if genre in nan_list:
        continue
    else:
        unique_genres.add(genre)

unique_genres = sorted(list(unique_genres))
for genre in unique_genres:
    print(genre)

acid jazz
acoustic pop
adult standards
afropop
alaska indie
album rock
alternative country
alternative dance
alternative hip hop
alternative metal
alternative pop
alternative pop rock
alternative rock
arkansas country
art pop
art rock
atl hip hop
australian alternative rock
australian americana
australian dance
australian indie folk
australian pop
australian psych
australian rock
austropop
barbadian pop
baroque pop
basshall
bebop
belgian pop
belgian rock
big beat
big room
blues
blues rock
bow pop
boy band
brill building pop
british alternative rock
british folk
british invasion
british singer-songwriter
british soul
britpop
bubblegum pop
canadian folk
canadian pop
canadian rock
candy pop
carnaval limburg
ccm
celtic
celtic punk
celtic rock
chamber pop
chanson
chicago soul
christelijk
classic canadian rock
classic country pop
classic italian pop
classic rock
classic schlager
classic soul
classic soundtrack
classic uk pop
classical rock
compositional ambient
contemporary country
contempor

##### **CONCLUSIONS:**
---
- **0 missing genre records** have been registered.

- **No inconsistencies** in genre naming found.

- **No cleaning procedures needed.**
---

#### 6) --BPM COLUMN INSPECTION--

- **Empty** and **unknown records** are to be looked for, counted and examined if present.

- BPM is numerical, so a safe **int() converstion will be performed to make sure it is numeric** with an exception block.

- **Suspicious values** will be checked for and examined if found (e.g. BPM value < 40 or > 240). The found values will be displayed along with the full ROW they belong to for a detailed inspection via external resources if needed.

##### Missing Records Scan (BEFORE CORRECTION)

In [21]:
missing_bpm_records = 0

for row in rows:
    bpm = row[6]
    
    try:
        bpm = bpm.lower()
        if bpm in nan_list:
            missing_bpm_records += 1
            print(f"\n< > Missing record\n|__VALUE: {value}\nTotal missing BPM records: {missing_bpm_records}")
    except ValueError:
        print(f"\n   <!> ValueError occurred:\n||__VALUE: {value}")
        continue
        
print("Total missing BPM records:", missing_bpm_records, calc_percent(missing_bpm_records))

Total missing BPM records: 0 (0.0%)


##### Numeric Value Validity Check (BEFORE CORRECTION)

In [22]:
for row in rows:
    bpm = row[6]
    
    try:
        bpm = int(bpm)
    except ValueError:
        print(f"\n   <!> ValueError occurred:\n|__VALUE: {value}")
        continue

##### Suspicious Values Scan (BEFORE CORRECTION)

In [23]:
sus_bpm_count = 0

for row in rows:
    bpm = int(row[6])

    if bpm < 40 or bpm > 240:
        sus_bpm_count += 1
        missing_bpm_records += 1
        print(f"\n<!> Suspicious values found:\n|__ROW: {row}\n|__VALUE: {bpm}")

print("\nTotal suspicious values:", sus_bpm_count, calc_percent(sus_bpm_count))
print("Total missing BPM records:", missing_bpm_records, calc_percent(missing_bpm_records))


<!> Suspicious values found:
|__ROW: ['1', 'Sunrise', 'Norah Jones', 'adult standards', '2004', '157', '30', '53', '-14', '11', '68', '201', '94', '3', 'UNKNOWN']
|__VALUE: 30

<!> Suspicious values found:
|__ROW: ['11', 'Love Me Tender', 'Elvis Presley', 'adult standards', '2002', '109', '5', '44', '-16', '11', '31', '162', '88', '4', '49']
|__VALUE: 5

<!> Suspicious values found:
|__ROW: ['19', 'Music', 'John Miles', 'classic uk pop', '2004', '87', '31', '27', '-13', '63', '12', '352', '1', '3', '46']
|__VALUE: 31

<!> Suspicious values found:
|__ROW: ['29', 'Der Weg', 'Herbert Grönemeyer', 'german pop', '2008', '142', '24', '34', '-11', '12', '19', '259', '92', '4', '48']
|__VALUE: 24

<!> Suspicious values found:
|__ROW: ['34', "Don't Know Why", 'Norah Jones', 'adult standards', '2002', '88', '20', '73', '-12', '7', '62', '186', '88', '3', '74']
|__VALUE: 20

<!> Suspicious values found:
|__ROW: ['40', 'De Weg', 'Guus Meeuwis', 'dutch pop', '2005', '72', '21', '46', '-11', '11', 

##### **CONCLUSIONS (BEFORE CORRECTION):**
---
- **0 missing genre records** have been registered.

- **No failed conversion cases** registered, implying that all the values are numeric.

- **416 suspicious values found** which makes **almost 21%** of the whole dataset row amount. The anomaly made me question the observed data and the correctness of my code. As the result, **the mistake in the code was found**. Particlularly, a **wrong column ID** was used to refer to the BPM column (`[6]` instead of `[5]`). **The corrected code follows:**
---

##### Missing Records Scan (AFTER CORRECTION)

In [24]:
missing_bpm_records = 0

for row in rows:
    bpm = row[5]
    
    try:
        bpm = bpm.lower()
        if bpm in nan_list:
            missing_bpm_records += 1
            print(f"\n< > Missing record\n|__VALUE: {bpm}\nTotal missing BPM records: {missing_bpm_records}")
    except ValueError:
        print(f"\n   <!> ValueError occurred:\n||__VALUE: {bpm}")
        continue
        
print("Total missing BPM records:", missing_bpm_records, calc_percent(missing_bpm_records))

Total missing BPM records: 0 (0.0%)


##### Numeric Value Validity Check (AFTER CORRECTION)

In [25]:
for row in rows:
    bpm = row[5]
    
    try:
        bpm = int(bpm)
    except ValueError:
        print(f"\n   <!> ValueError occurred:\n|__VALUE: {bpm}")
        continue

##### Suspicious Values Scan (AFTER CORRECTION)

In [26]:
sus_bpm_count = 0

for row in rows:
    bpm = int(row[5])

    if bpm < 40 or bpm > 240:
        sus_bpm_count += 1
        missing_bpm_records += 1
        print(f"\n<!> Suspicious values found:\n|__ROW: {row}\n|__VALUE: {bpm}")

print("\nTotal suspicious values:", sus_bpm_count, calc_percent(sus_bpm_count))
print("Total missing BPM records:", missing_bpm_records, calc_percent(missing_bpm_records))


<!> Suspicious values found:
|__ROW: ['999', 'Still Crazy After All These Years', 'Paul Simon', 'classic rock', '1975', '37', '25', '27', '-12', '9', '13', '207', '80', '4', '61']
|__VALUE: 37

Total suspicious values: 1 (0.05%)
Total missing BPM records: 1 (0.05%)


##### **CONCLUSIONS (AFTER CORRECTION):**
---
- **0 missing genre records** have been registered.

- **No failed conversion cases** registered, implying that all the values are numeric.

- **1 suspicious value registered** and explored further. Although the value is beyond the common BPM range, this still **can be a legitimate musical interpretation** (half-time interpretation vs. double-time interpretation). Both can be legitimate, and the found extreme value could be converted into the common range via doublind (37 * 2 = 74). However, **the original value will be preserved** for this use case.
---

#### 7) --POPULARITY COLUMN INSPECTION--

- **Empty** and **unknown records** are to be looked for, counted and examined if present.

- Popularity is numerical. Safe **int() converstion will be performed to make sure it is numeric** with an exception block.

- The dataset features popularity index from 0 to 100. The column will be **scanned for out-of-bounds values** (e.g. values less than 0 and more than 100).

##### Missing Records Scan

In [27]:
missing_pop_records = 0

for row in rows:
    pop = row[14]
    try:
        pop = pop.lower()
        if pop in nan_list:
            missing_pop_records += 1
            print(f"\n< > Missing record\n|__VALUE: {pop}\nTotal missing popularity records: {missing_pop_records}")
    except ValueError:
        print(f"\n   <!> ValueError occurred:\n   ||__VALUE: {pop}")
        continue

print("\nTotal missing popularity records:", missing_pop_records, calc_percent(missing_pop_records))


< > Missing record
|__VALUE: unknown
Total missing popularity records: 1

< > Missing record
|__VALUE: 
Total missing popularity records: 2

< > Missing record
|__VALUE: unknown
Total missing popularity records: 3

< > Missing record
|__VALUE: 
Total missing popularity records: 4

< > Missing record
|__VALUE: 
Total missing popularity records: 5

< > Missing record
|__VALUE: unknown
Total missing popularity records: 6

< > Missing record
|__VALUE: 
Total missing popularity records: 7

< > Missing record
|__VALUE: 
Total missing popularity records: 8

< > Missing record
|__VALUE: unknown
Total missing popularity records: 9

< > Missing record
|__VALUE: 
Total missing popularity records: 10

< > Missing record
|__VALUE: unknown
Total missing popularity records: 11

< > Missing record
|__VALUE: 
Total missing popularity records: 12

< > Missing record
|__VALUE: unknown
Total missing popularity records: 13

< > Missing record
|__VALUE: unknown
Total missing popularity records: 14

< > Mis

##### Numeric Value Validity Check

In [28]:
missing_pop_records = 0

for row in rows:
    pop = row[14]

    try:
        pop = pop.lower()
        if pop in nan_list:
            missing_pop_records += 1
            continue
        else:
            try:
                pop = int(pop)
            except ValueError:
                missing_pop_records += 1
                print(f"\n< > Missing record\n|__VALUE: {pop}\nTotal missing popularity records: {missing_pop_records}")
                continue
    except ValueError as er:
        print(er)
        continue


##### Out-of-bounds Value Scan

In [29]:
for row in rows:
    pop = row[14]

    try:
        pop = pop.lower()
        if pop in nan_list:
            continue
        else:
            try:
                pop = int(pop)
                if 0 < pop <100:
                    continue
                else:
                    missing_pop_records += 1
                    print(f"\n< > Out-of-bounds value\n|__VALUE: {pop}\nTotal missing popularity records: {missing_pop_records}")
            except ValueError:
                continue
    except ValueError as er:
        print(er)
        continue

print("\nTotal missing popularity records:", missing_pop_records, calc_percent(missing_pop_records))


< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 202

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 203

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 204

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 205

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 206

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 207

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 208

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 209

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 210

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 211

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 212

< > Out-of-bounds value
|__VALUE: 999999
Total missing popularity records: 213

< > Out-of-bounds value
|__VALUE: 99999

##### **CONCLUSIONS:**
---
- **245 missing popularity records found** (including out-of-bounds sentinal values) (12.29% of total data).

- **No failed conversion cases registered**, implying that all the values are numeric.

- Due to the fact that **missing populartiy records make 12.29%** of total data, a thorough consideration was made regarding the further actions toward such a large fraction of the total dataset. Deleting the rows would also delete valuable information stored in other columns which could be used in other calculations. While filling the missing values with calculations like the mean or, rather, median of a given artist's other tracks popularity score would violate the metric inregrity through artificial assumptions. Eventually, **the decision was made to exclude the rows** containing missing popularity records from the downstream mathematical calculations **without removing them** from the dataset.

- As the result, **any incomplete row** in the dataset **will be preserved** and **any missing record will be converted into Python `None` value** to avoid information loss and preserve data transparency. If the dataset was not broken synthetically, and if the original, cleaner version of it did not exist, the resulting cleaned dataset could be exported as a separate file and stored for any other potential use cases.

---

#### **DATA INSPECTION SUMMARY**

- **50 duplicated rows found** and **removed from the dataset successfully** as shown by performed validation checks.

- **318 missing entries found** in total (15.95% of the subset's total data). This includes **73 missing Artist entries** and **245 missing Popularity entries**.

- **The rows with missing Artist entries will be excluded from the final dataset** due to the Artist value being the central value of the defined data objective.

- **The rows with missing Popularity entries will be excluded from the downstream calculations** but kept within the dataset to preserve the remaining information. The missing values will be converted into Python `None` values.

---
---

### STAGE 3 - DATA CLEANING & TRANSFORMATION


#### 1) --DEFINING DATA CLEANING OBJECTIVES--

- The conclusions made during the data inspection stage define actions which are to be performed during the downstream data cleaning stage. **The data cleaning objectives are:**

    - Exclude irrelevant columns from data cleaning process. **The columns that will be processed** are as follows:
        - Artist
        - Top Genre
        - BPM
        - Popularity
      <br>
          
    - **Convert all missing values (318 total)** in the targeted columns **into Python `None` value** for data preservation and dataset transparency. This also includes out-of-bounds sentinal values in Popularity column.
      
    - **Convert all numeric data** in the featured columns **into numeric type**.
      
    - **Skip rows with missing Artist entries** during the cleaning process, due to the Artist value being the foundation value of the final dataset.
 
    - **Clean Artist and Top Genre values** in a standard way (remove extra spaces, standardize casing).
      
    - **Exclude missing Popularity values from mathematical calculations** of average popularity per artist.
 
    - **Create a pre-final (pre-aggregation) vesrion dataset of dictionaris** of the following structure:
    ```
          {
           artist name: {
                    Top genre list: []
                    BPM list: []
                    Popularity list: []
                   }
          }
   
    ```

|
---

#### 2) --FUNCTIONS--

- During the cleaning process, predesigned functions will be applied for:
    - **String cleaning** (standardizing casing and removing extra spaces).
    - **Converting missing values into `none` type**.
    - **Converting out-of-bounds Popularity values into `None` type**.
    - **Converting data in the numeric columns** (BPM, Popularity) **into `int` type**.

##### String Clean Function

In [30]:
def str_clean(value):
    if value and type(value) == str:
        value = value.strip().lower().title()
    return value

##### Turn None Function

In [31]:
def turn_none(value):
    nan_list = ['unknown', 'null', 'n/a', 'none', '']
    if not value:
        value = None
    elif (type(value) == str) and value.lower() in nan_list:
        value = None
    return value  

##### Turn Integer Function

In [32]:
def turn_int(value):
    if value and value.isdigit():
        value = int(value)
    else:
        value = None
    return value

##### Turn None Function (for out-of-bounds Popularity values spiecifically)

In [33]:
def turn_none_pop(value):
    if value and (type(value) == int) and not (0 < int(value) < 100):
        value = None
    return value

---

#### 3) --DATA CLEANING--

1) **Skip rows with missing Artist entries**.

2) Skip irrelevant columns. **The columns relevant to the data objective are:**
    - **Artist** (column ID: `[2]`)
    - **Top Genre** (column ID: `[3]`)
    - **BPM** (column ID: `[5]`)
    - **Popularity** (column ID: `[14]`)
    
    <br>

3) **Clean string data** (standardize casing and remove extra spaces) with `str_clean` function.

4) **Convert missing values into `None`** with `turn_none` function.

5) **Convert numeric data into `integer` type** with `turn_int` function.

6) **Convert out-of-bounds Popularity values into `None`** with `turn_none_pop` function.

7) **Create a pre-final (pre-aggregation) vesrion dataset of dictionaris** of the following structure:
    ```
          {
           artist name: {
                    Top genre list: []
                    BPM list: []
                    Popularity list: []
                   }
          }
   
    ```
    

##### Row-by-Row Cleaning Process

In [34]:
dic_prefinal = {}

for i, row in enumerate(rows):
    
    if i in drop_row_id_list:  #1
        continue
    if row[0] in missing_artist_row_id:  #2
        continue

    artist = turn_none(str_clean(row[2]))  #3, #4 (although missing artist entries should have been filtered out, #4 is applied as a precaution)
    genre = turn_none(str_clean(row[3]))  #3, #4 (although Top Genre column did not feature missing entries, #4 is applied as a precaution)
    bpm = turn_int(turn_none(str_clean(row[5])))  #4, #5
    pop = turn_none_pop(turn_int(turn_none(str_clean(row[14]))))  #4, #5, #6

    dic_prefinal[artist] = dic_prefinal.get(artist, {  #7
        "Top genre list": [],
        "BPM list": [],
        "Popularity list": []        
    })

    dic_prefinal[artist]["Top genre list"].append(genre)
    dic_prefinal[artist]["BPM list"].append(bpm)
    dic_prefinal[artist]["Popularity list"].append(pop)

for i, (key, val) in enumerate(dic_prefinal.items()):
    print(f"{i+1} {key}:")
    for k, v in val.items():
        print(f"   |{k}: {v}")

1 Norah Jones:
   |Top genre list: ['Adult Standards', 'Adult Standards']
   |BPM list: [157, 88]
   |Popularity list: [None, 74]
2 Deep Purple:
   |Top genre list: ['Album Rock', 'Album Rock', 'Album Rock', 'Album Rock']
   |BPM list: [135, 127, 114, 127]
   |Popularity list: [39, 45, 64, 37]
3 Gorillaz:
   |Top genre list: ['Alternative Hip Hop', 'Alternative Hip Hop']
   |BPM list: [168, 139]
   |Popularity list: [69, 80]
4 Foo Fighters:
   |Top genre list: ['Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal']
   |BPM list: [173, 168, 145, 130, 144, 138, 135, 158, 136]
   |Popularity list: [76, 72, 65, 75, 69, 69, 64, 77, None]
5 Bruce Springsteen:
   |Top genre list: ['Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classi

##### **CONCLUSIONS:**
---
- **711 Artist entries have been added** to the pre-final dictionary **along with the targeted data** (per artist entry):
    - a list of top genres,
    - a list of BPM values,
    - a list of popularity values.
    <br>

- The resulting dictionary is thus being ready for the transformation process.
---

#### 4) --DATA TRANSFORMATION (ABORTED)--

- **Data aggregation will be performed** for each Artist entry, specifically:
    - Top genre (the most frequently featured top genre per artist)
    - Average BPM (beats per minute)
    - Average popularity (`None` values will be excluded from the calculation)
    <br>

- Identifying the most frequently featured top genre per artist, **a dictionary will be constructed to store genre counts** for each artist. If a given artist has more than one top genre in their list of top genres, **the most frequent one will be retreived to be stored in the final dataset**.
    

##### Data Aggregation Process (Final Dataset Dictionary) (Aborted)

In [35]:
dic_final = {}

for artist, dic in dic_prefinal.items():
    
    dic_final[artist] = artist

    genre_dic = {}
    for genre in dic["Top genre list"]:
        genre_dic[genre] = genre_dic.get(genre, 0) + 1
    genre_ls = list(genre_dic.items())
    genre_ls.sort(reverse=False)
    if len(genre_ls) > 1:
        print("   ", artist)
        print(genre_ls)
  
        

    Nan
[('Album Rock', 2), ('Alternative Metal', 1), ('Alternative Rock', 3), ('Baroque Pop', 1), ('Brill Building Pop', 1), ('British Soul', 1), ('Classic Rock', 1), ('Dance Pop', 1), ('Dutch Hip Hop', 1), ('Dutch Pop', 1), ('Irish Rock', 1), ('Modern Rock', 1), ('Pop', 1)]


##### **CONCLUSIONS:**
---
- During the process of identifying the most frequently featured top genre per artist, **a validation check was performed** which **allowed to discover an oversight related to the data inspection stage**.

- **Only 1 artist entry featured more than 1 unique top genre** (16 genres listed in total, 13 unique genres listed in total) in their list of top genres **which seems suscpicious**. Looking at the Artist value ("Nan") it becomes clear that a sentinel value have been identified which was not included into the list of sentinel values in the beginning of the data inspection process.

- However, it is worth mentioning, that despite "nan" being a typical sentinel value for a missing record, even **such a name could potentially be an actual name of an established artist**. The external reference revealed that indeed several artists with such a name do exist and are broadcast on Spotify. Thus, the most honest decision would be to return to each "nan" artist entry within the dataset along with the track titles associated with it and release year and **check the artist-title-year combination by means of additional research and external reference in an attempt to identify if any such relationship does exist**. This will be performed shortly.

- Finally, depending on the results of the research, **the cleaning rules will be updated** and **the pre-final dictionary complied anew**.

---

#### 5) --EMERGENCY DATA CHECK--
    

##### Artist-Title-Year Relationship Retrieval

In [36]:
count = 0

for i, row in enumerate(rows):
    artist = row[2]
    title = row[1]
    year = row[4]
    
    if artist == "nan" or artist == "NaN" or artist == "Nan" or artist == "NAN":
        count += 1
        print(f"{count} (Row ID: {i}) {artist} - {title} ({year})")

1 (Row ID: 118) nan - Use Somebody (2008)
2 (Row ID: 213) nan - Can't Stop (2002)
3 (Row ID: 257) nan - Hometown Glory (2008)
4 (Row ID: 432) nan - Hold On Tight (2010)
5 (Row ID: 507) nan - Slapeloze Nachten (2012)
6 (Row ID: 655) nan - CAN'T STOP THE FEELING! (Original Song from DreamWorks Animation's "TROLLS") (2016)
7 (Row ID: 661) nan - Bridges (2016)
8 (Row ID: 787) nan - Memories (2019)
9 (Row ID: 836) nan - Tears In The Morning - Remastered 2009 (1970)
10 (Row ID: 1062) nan - You Took The Words Right Out of My Mouth (Hot Summer Night) (1977)
11 (Row ID: 1275) nan - Fade To Black (Remastered) (1984)
12 (Row ID: 1564) nan - Mysterious Ways (1991)
13 (Row ID: 1571) nan - Kayleigh (1992)
14 (Row ID: 1594) nan - Find The River (1992)
15 (Row ID: 1804) nan - Scar Tissue (1999)
16 (Row ID: 1839) nan - The Sound of Silence - Acoustic Version (1964)


##### **CONCLUSIONS:**
---
- **16 Artist-Title-Year relationships were checked**.

- The performed external research **did not confirm the existence of such Artist-Title-Year relationships**.

- **"nan" Artist entry will be added to the list of sentinel values**, the **`turn_nan` function updated**, and **the pre-final dictionary recompiled**, excluding "nan" Artist entries from its structure.

- With "nan" Artist entries absent, **no Artist entries within the dictionary will feature more than 1 unique genre in their genre list**, implying the lack of need to compile an additional, internal dictionary of genres to count unique genre values per artist and retrieve the most frequently featured one for the final dataset with aggregated information. Instead, **any value from the genre list per Artist can be used as the Top Genre value in the final dataset**.]

---

#### 6) --DATA CLEANING (UPDATED & REINITIATED)--
    

##### Turn None Function Update

In [37]:
def turn_none(value):
    nan_list = ['unknown', 'null', 'n/a', 'none', '', 'nan']  # Added 'nan' sentinel value to the list.
    if not value:
        value = None
    elif (type(value) == str) and value.lower() in nan_list:
        value = None
    return value  

##### Row-by-Row Cleaning Process Update

In [38]:
dic_prefinal = {}

for i, row in enumerate(rows):
    
    if i in drop_row_id_list:
        continue
    if row[0] in missing_artist_row_id:
        continue

    artist = turn_none(str_clean(row[2]))
    genre = turn_none(str_clean(row[3]))
    bpm = turn_int(turn_none(str_clean(row[5])))
    pop = turn_none_pop(turn_int(turn_none(str_clean(row[14]))))

    if artist:  # Skipping `None` Artist values.
        dic_prefinal[artist] = dic_prefinal.get(artist, {
            "Top genre list": [],
            "BPM list": [],
            "Popularity list": []        
        })
    
        dic_prefinal[artist]["Top genre list"].append(genre)
        dic_prefinal[artist]["BPM list"].append(bpm)
        dic_prefinal[artist]["Popularity list"].append(pop)

for i, (key, val) in enumerate(dic_prefinal.items()):
    print(f"{i+1} {key}:")
    for k, v in val.items():
        print(f"   |{k}: {v}")

1 Norah Jones:
   |Top genre list: ['Adult Standards', 'Adult Standards']
   |BPM list: [157, 88]
   |Popularity list: [None, 74]
2 Deep Purple:
   |Top genre list: ['Album Rock', 'Album Rock', 'Album Rock', 'Album Rock']
   |BPM list: [135, 127, 114, 127]
   |Popularity list: [39, 45, 64, 37]
3 Gorillaz:
   |Top genre list: ['Alternative Hip Hop', 'Alternative Hip Hop']
   |BPM list: [168, 139]
   |Popularity list: [69, 80]
4 Foo Fighters:
   |Top genre list: ['Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal', 'Alternative Metal']
   |BPM list: [173, 168, 145, 130, 144, 138, 135, 158, 136]
   |Popularity list: [76, 72, 65, 75, 69, 69, 64, 77, None]
5 Bruce Springsteen:
   |Top genre list: ['Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classic Rock', 'Classi

##### **CONCLUSIONS:**
---
- With "nan" Artist values excluded, **710 Artist entries have been added** to the pre-final dictionary.

- The dictionary is ready for the transformation stage.
---

#### 7) --DATA TRANSFORMATION (UPDATED & CONTINUED)--
    

##### Data Aggregation Process (Continued)

In [39]:
dic_final = {}

for artist, dic in dic_prefinal.items():
    
    top_genre = dic["Top genre list"][0]
    bpm_total = 0
    pop_total = 0
    
    for bpm_val in dic["BPM list"]:
        bpm_total += bpm_val
    bmp_avg = round(bpm_total / len(dic["BPM list"]))

    for pop_val in dic["Popularity list"]:
        if pop_val:
            pop_total += pop_val
    if pop_total > 0:
        pop_avg = round(pop_total / len(dic["Popularity list"]))
    
    dic_final[artist] = artist
    
    dic_final[artist] = {
        "Top genre": top_genre,
        "Avg BPM": bmp_avg,
        "Avg popularity": pop_avg
    }

for i, (artist, dic) in enumerate(dic_final.items()):
    print(f"{i + 1} {artist}:")
    for key, val in dic.items():
        print(f"   |{key}: {val}")
        

1 Norah Jones:
   |Top genre: Adult Standards
   |Avg BPM: 122
   |Avg popularity: 37
2 Deep Purple:
   |Top genre: Album Rock
   |Avg BPM: 126
   |Avg popularity: 46
3 Gorillaz:
   |Top genre: Alternative Hip Hop
   |Avg BPM: 154
   |Avg popularity: 74
4 Foo Fighters:
   |Top genre: Alternative Metal
   |Avg BPM: 147
   |Avg popularity: 63
5 Bruce Springsteen:
   |Top genre: Classic Rock
   |Avg BPM: 120
   |Avg popularity: 51
6 City To City:
   |Top genre: Alternative Pop Rock
   |Avg BPM: 99
   |Avg popularity: 45
7 Maroon 5:
   |Top genre: Pop
   |Avg BPM: 102
   |Avg popularity: 74
8 Muse:
   |Top genre: Modern Rock
   |Avg BPM: 130
   |Avg popularity: 63
9 The Killers:
   |Top genre: Modern Rock
   |Avg BPM: 140
   |Avg popularity: 73
10 Eminem:
   |Top genre: Detroit Hip Hop
   |Avg BPM: 116
   |Avg popularity: 65
11 Elvis Presley:
   |Top genre: Adult Standards
   |Avg BPM: 120
   |Avg popularity: 52
12 The White Stripes:
   |Top genre: Alternative Rock
   |Avg BPM: 124
   |Avg

##### **CONCLUSIONS:**
---
- **The aggregation process perfromed on 710 Artist entries**.

---

#### **DATA CLEANING & TRANSFORMATION SUMMARY**

- A **subset of the initial dataset was selected** for processing, limited to:

    - Rows with valid artist entries,
    - Targeted columns:
  
        - Artist
        - Top Genre
        - BPM
        - Popularity
    <br>

- Within the subset, **following cleaning procedures** were performed during the data cleaning process:

    - 318 missing values converted into Python `None` value
    - String data cleaned in a standard way (extra spaces removed, casing standardized)
    - Numeric data converted into `int` type
    - Missing Popularity values excluded from mathematical calculations
    <br>

- The cleaned data was **organized into a dictionary** ready for the transformation process.

- During the transformation process, one **data cleaning mistake was identified**:

    - **'nan' sentinel value** within the Artist column was missed by an oversight which led to 'Nan' artist value being added to the resulting dictionary as a valid artist entry (which eventually was proved wrong).
    <br>

- The data cleaning procedures had to be updated and the dictionary reassambled.

- After the code correction, the transformation process was reiniteated, and **aggregation calculations were performed** per each Artist entry:

    - Top Genre per artist
    - Average BPM per artist
    - Average popularity per artist.
    <br>

- **710 Artist entries** were processed and added to the final dictionary.

- Next, the dictionary is to be subjected to **data validation checks** to ensure data integrity.

---

### STAGE 4 - DATA VALIDATION


- **A number of validation chekcs will be performed** on the resulting dictionary before the final dataset export.

- **An arbitrary Artist entry will be selected and retreived from the original dataset**, its data inspected and the average numbers recalculated, to be then **compared with the corresponding entry within the final dictionary**.

#### --Selective Data Validation Check #1: "Muse"

In [40]:
value = "Muse"
print(value)
print()

count = 0
bpm_count = 0
pop_count = 0
genres = []
bpm_total = 0
pop_total = 0

for row in rows:
    artist = row[2]
    genre = row[3]
    bpm = row[5]
    pop = row[14]
    
    if artist.strip() == value:
        count += 1
        genres.append(genre)
        if bpm and bpm.isdigit():
            bpm_count += 1
            bpm = int(bpm)
            bpm_total += bpm
        if pop and pop.isdigit():
            pop_count += 1
            pop = int(pop)
            pop_total += pop
        
        print(row)

bpm_avg = round(bpm_total / bpm_count)
pop_avg = round(pop_total / pop_count)

print()
print("| Featured genres:", genres)
print("| Average BPM:", bpm_avg)
print("| Aberage popularity:", pop_avg)

if value in dic_final:
    print()
    print("Dictionary record found:", value)
    print(dic_final[value])
else:
    print()
    print(f"<< Dictionary record not found: {value} >>")


Muse

['8', 'Knights of Cydonia', 'Muse', 'modern rock', '2006', '137', '96', '37', '-5', '12', '21', '366', '0', '14', '69']
['117', 'Feeling Good', 'Muse', 'modern rock', '2001', '109', '42', '35', '-8', '10', '27', '199', '30', '3', '51']
['165', 'Uprising', 'Muse', 'modern rock', '2009', '128', '91', '60', '-4', '12', '41', '305', '0', '8', '76']
['189', 'Plug in Baby', 'Muse', 'modern rock', '2001', '136', '97', '41', '-4', '11', '35', '218', '0', '5', '53']
['198', 'Starlight', 'Muse', 'modern rock', '2006', '122', '87', '55', '-4', '21', '32', '240', '0', '3', '73']
['204', 'Supermassive Black Hole', 'Muse', 'modern rock', '2006', '120', '92', '67', '-4', '9', '78', '212', '5', '4', '73']
['255', 'Time Is Running Out', 'Muse', 'modern rock', '2004', '118', '84', '59', '-6', '9', '43', '237', '0', '6', '69']
['315', '  Sing for Absolution  ', 'Muse', 'modern rock', '2004', '170', '68', '44', '-7', '9', '19', '295', '46', '3', '57']
['339', 'Resistance', 'Muse', 'modern rock', '20

##### **CONCLUSION:**
---
- The first arbitrary check **shows consistency** between the entry retrieved from the original dataset and the corresponding one retreived from the final dictionary.
---

#### --Selective Data Validation Check #2: "AC/DC"

In [41]:
value = "AC/DC"
print(value)
print()

count = 0
bpm_count = 0
pop_count = 0
genres = []
bpm_total = 0
pop_total = 0

for row in rows:
    artist = row[2]
    genre = row[3]
    bpm = row[5]
    pop = row[14]
    
    if artist.strip() == value:
        count += 1
        genres.append(genre)
        if bpm and bpm.isdigit():
            bpm_count += 1
            bpm = int(bpm)
            bpm_total += bpm
        if pop and pop.isdigit():
            pop_count += 1
            pop = int(pop)
            pop_total += pop
        
        print(row)

bpm_avg = round(bpm_total / bpm_count)
pop_avg = round(pop_total / pop_count)

print()
print("| Featured genres:", genres)
print("| Average BPM:", bpm_avg)
print("| Aberage popularity:", pop_avg)

if value in dic_final:
    print()
    print("Dictionary record found:", value)
    print(dic_final[value])
else:
    print()
    print(f"<< Dictionary record not found: {value} >>")


AC/DC

['1036', 'Whole Lotta Rosie', 'AC/DC', 'album rock', '1977', '159', '84', '29', '-4', '10', '42', '334', '0', '8', '65']
['1054', 'Let There Be Rock', 'AC/DC', 'album rock', '1977', '91', '73', '49', '-6', '8', '43', '366', '0', '9', '']
['1114', 'Highway to Hell', 'AC/DC', 'album rock', '1979', '116', '91', '57', '-5', '16', '42', '', '6', '13', '83']
['1158', 'Back In Black', 'AC/DC', 'album rock', '1980', '188', '70', '31', '-6', '8', '76', '', '1', '5', '83']
['1161', 'You Shook Me All Night Long', 'AC/DC', 'album rock', '1980', '127', '77', '53', '-6', '39', '76', '210', '0', '6', '']
['1497', 'Thunderstruck', 'AC/DC', 'album rock', '1990', '134', '89', '50', '-5', '22', '26', '293', '0', '4', '81']

| Featured genres: ['album rock', 'album rock', 'album rock', 'album rock', 'album rock', 'album rock']
| Average BPM: 136
| Aberage popularity: 78

<< Dictionary record not found: AC/DC >>


##### **CONCLUSION:**
---
- **The second arbitrary check fails**. The key "AC/DC" was not found within the final dictionary which **highlights the critical oversight in the approach to the cleaning process**. Specifically, **not taking into account special string casing** that is common within artist names. Thus, names such as "AC/DC", "OneRepublic", and others **are deprived of their signature casing during the data standardizing procedure**.
 
- String `lower()` and `title()` methods will be applied to the selected value and the search within the final dictionary will be performed again to confirm the issue.
---

##### Casing Loss Problem Check

In [42]:
value = "AC/DC"
value = value.lower().title()

if value in dic_final:
    print()
    print("Dictionary record found:", value)
    print(dic_final[value])
else:
    print()
    print(f"<< Dictionary record not found: {value} >>")


Dictionary record found: Ac/Dc
{'Top genre': 'Album Rock', 'Avg BPM': 136, 'Avg popularity': 52}


##### **CONCLUSION:**
---
- **The casing loss problem was confirmed**. The string methods changed the original capitalized name from "AC/DC" into "Ac/Dc" which points at the oversight concerning the string cleaning procedure.
 
- Thus, **the string cleaning procedure will have to be reviewed and redesigned** to avoid losing critical information in those cases when artist names feature specific casing.

- Additionally, successful retrieval of the entry from the final dictionary reveals that **average popularity results do not match** between the calculations performed on the original data and those stored within the final dictionary data, which is worth inspecting as well.
---

##### Popularity Calculation Inspection

In [43]:
value = "AC/DC"
print(value)
print()

count = 0
bpm_count = 0
pop_count = 0
genres = []
bpm_total = 0
pop_total = 0

for row in rows:
    artist = row[2]
    genre = row[3]
    bpm = row[5]
    pop = row[14]
    
    if artist == value:
        print(row)
        count += 1
        genres.append(genre)
        if bpm and bpm.isdigit:
            bpm_count += 1
            bpm = int(bpm)
            bpm_total += bpm
        if pop and pop.isdigit:
            pop_count += 1
            pop = int(pop)
            pop_total += pop
            print("Popularity value count:", pop_count)
            print("Popularity value:", pop)
            print()

bpm_avg = round(bpm_total / bpm_count)
pop_avg = round(pop_total / pop_count)

print("| Featured genres:", genres)
print("| Average BPM:", bpm_avg)
print("| Aberage popularity:", pop_avg)
print()


value = value.lower().title()

if value in dic_final:
    print("---")
    print("Dictionary (Final) record found:", value)
    print(dic_final[value])
else:
    print("---")
    print(f"<< Dictionary record not found: {value} >>")


if value in dic_prefinal:
    print("---")
    print("Dictionary (Prefinal) record found:", value)
    print(dic_prefinal[value])
else:
    print("---")
    print(f"<< Dictionary record not found: {value} >>")



AC/DC

['1036', 'Whole Lotta Rosie', 'AC/DC', 'album rock', '1977', '159', '84', '29', '-4', '10', '42', '334', '0', '8', '65']
Popularity value count: 1
Popularity value: 65

['1054', 'Let There Be Rock', 'AC/DC', 'album rock', '1977', '91', '73', '49', '-6', '8', '43', '366', '0', '9', '']
['1114', 'Highway to Hell', 'AC/DC', 'album rock', '1979', '116', '91', '57', '-5', '16', '42', '', '6', '13', '83']
Popularity value count: 2
Popularity value: 83

['1158', 'Back In Black', 'AC/DC', 'album rock', '1980', '188', '70', '31', '-6', '8', '76', '', '1', '5', '83']
Popularity value count: 3
Popularity value: 83

['1161', 'You Shook Me All Night Long', 'AC/DC', 'album rock', '1980', '127', '77', '53', '-6', '39', '76', '210', '0', '6', '']
['1497', 'Thunderstruck', 'AC/DC', 'album rock', '1990', '134', '89', '50', '-5', '22', '26', '293', '0', '4', '81']
Popularity value count: 4
Popularity value: 81

| Featured genres: ['album rock', 'album rock', 'album rock', 'album rock', 'album rock

##### **CONCLUSION:**
---
- The displayed code has **no mistake in the code processing the original data.**
 
- Instead, the mistake **was found in the calculations within the pre-final dicitonary to final dictionary** code.

- **The problem was in using `len(dic["Popularity list"])` as the divider** to calculate the average of popularity values. Although the popularity total sum successfuly excluded the `None` values from its calculation, **the `None` values were still present within the "Popularity list"** which was used as a divider. As the result, **the incorrect number in the devider skewed the calculation results**.

In [ ]:
# Snippet: Incorrect code

for pop_val in dic["Popularity list"]:
    if pop_val:
        pop_total += pop_val
if pop_total > 0:
    pop_avg = round(pop_total / len(dic["Popularity list"]))  # Popularity list with None values used as the divider

In [ ]:
# Snippet: Corrected code 

for pop_val in dic["Popularity list"]:
    if pop_val:
        pop_val_count +=1  # Value count will be used instead of the list length to exclude None values
        pop_total += pop_val
if pop_total > 0:
    pop_avg = round(pop_total / pop_val_count)

- **The cleaning and transformation process will have to be updated** with the corrected code **and performed again**.

---

#### **DATA VALIDATION SUMMARY**

- As a data validation check, **an arbitrary Artist entry was be selected and retreived from the original dataset**, its data inspected and the average numbers recalculated, to be then **compared with the corresponding entry within the final dictionary**.

- One of such validation checks **revealed two critical problems**:

    - **Artist name signature casing loss** during the string data cleaning procedures due to the `lower()` and `title()` methods  application on all the processed string data.
    - **A mistake in the average popularity calculation** due to the divider contaning `None` Popularity values which had to be excluded.
    <br>

- Thus, **the identified problems will have to be faced**, the solutions found, and the data inspection, cleaning, and **the transformation processes reviewed, corrected, and reinitiated** again.

---
---

### STAGE 5 - POST-VALIDATION DATA CLEANING & TRANSFORMATION APPROACHES CORRECTION

#### 1) --DEFINING THE PROBLEMS & SOLUTIONS--


**PROBLEM 1:**

- **Artist name signature casing loss** during the string data cleaning procedures due to the `lower()` and `title()` methods  application on all the processed string data.

**PROBLEM 2:**

- **Incorrect divider value (`len(dic["Popularity list"])`) in the average population calculation code** due to the oversight which allowed `None` Popularity values being included in the calculation.

    <br>

**SOLUTION TO PROBLEM 1:**

- Due to **the major reasons behind applying the `lower()` and `title()` methods** was using them as one of the measures against accidental string duplication (due to potentially improper casing), as well as for sentinel value identification, the rules for performing these procedures will have to be reviewed and redesigned.

- Thus, **instead of permanently applying the methods** to the processed string value, **the methods will be applied only temporarily during the checks** for duplication and sentinal values. After the check is finished, in case of no duplication or sentinel value instance, the value will be stored with the original casing. If the value is a sentinel value or a duplicate, it will be marked as `None`.

- The risk remains that some inconsistent casing is featured in the original dataset itself, and thus may be included into the final dataset. For cases like this, perhaps, a mapping dictionary of correct artist/track names (or exceptions) could be created and actively updated to be applied to each artist/track name value after data cleaning.

**SOLUTION TO PROBLEM 2:**

- **A dedicated counter will be added** to the Popularity values parsing code to count total number of valid Popularity entries and its final value **used as the divider** in the average popularity calculation **instead of the `len(dic["Popularity list"])`**.

    <br>
  
**ADDITIONAL IMPROVEMENTS:**

- **Each case of the Artist column containing any of the listed sentinel values as its artist name** (except for `''` and 'nan') **will be manually inspected** instead of automatically converted into `None` **due to the risk of the value being not a sentinel one but a real established artist name**.

---

#### 2) --DATA INSPECTION (UPDATED)--

- **Each listed sentinel value within the Artist column** will be **inspected to differentiate an actual sentinel value from a potentially real artist name** by **checking the artist-title-year combination via external reference.**

- Two sentinel values will be skipped during the inspection:

    - `''` (empty string)
    - 'nan' (which has already been inspected and no "Nan" was registered to represent a real artist name in this subset of data)

##### Artist-Title-Year Relationship Retrieval

In [45]:
nan_list = ['unknown', 'null', 'n/a', 'none', '', 'nan']
count1 = 0
count2 = 0

for i, row in enumerate(rows):
    artist = row[2]
    title = row[1]
    year = row[4]
    
    if any(substring == artist.strip().lower() for substring in nan_list):
        if artist.strip().lower() == '' or artist == 'nan':
            count1 += 1
            continue
        else:
            count2 += 1
            print(f"{count2} {artist} - {title} ({year})")

print("Total:", count1 + count2)

1 None - Iris (2007)
2 UNKNOWN - In The Army Now (2002)
3 NULL - Maybe Tomorrow (2003)
4 UNKNOWN - No One Knows (2002)
5 None - Ne me quitte pas (2004)
6 UNKNOWN - One Day Like This (2009)
7 None - Love Is A Battlefield (2001)
8 None - father & friend (2007)
9 NULL -   Stuck In A Moment You Can't Get Out Of   (2000)
10 UNKNOWN - Time To Say Goodbye (Con Te Partirò) (2006)
11 None - De Kapitein Deel II (2000)
12 None - woman (2006)
13 None - Mother Earth (2003)
14 NULL -   Back Down South   (2010)
15 UNKNOWN - tubular bells (2012)
16 NULL - Little Talks (2012)
17 NULL - Chandelier (2015)
18 None -   The Less I Know The Better   (2015)
19 NULL - Love Yourself (2015)
20 NULL - dark necessities (2016)
21 NULL - in the blood (2017)
22 None - Have You Ever Seen The Rain (1970)
23 UNKNOWN - Carry On (1970)
24 UNKNOWN - Baba O'Riley (1971)
25 None - Nothing Rhymed (1971)
26 None - Locomotive Breath (1971)
27 NULL - Changes - 2015 Remaster (1971)
28 UNKNOWN - A Horse with No Name (1972)
29 UNKN

##### **CONCLUSION:**
---
- **55 Artist-Title-Year relationships checked**.

- The performed external referencing **did not confirm the existence of such Artist-Title-Year relationships**. Therefore, all the sentinel-featuring Artist entries will be considered missing entries.
 
- The **missing Artist entries will be excluded** from the cleaning and transformation stage.
---

#### 3) --DATA CLEANING & TRANSFORMATION (UPDATED 1)--

- **The earlier established data cleaning procedures will be performed** (with minor change in step 3):

    1) **Skip rows with missing Artist entries**.
    
    2) Skip irrelevant columns. **The columns relevant to the data objective are:**
        - **Artist** (column ID: `[2]`)
        - **Top Genre** (column ID: `[3]`)
        - **BPM** (column ID: `[5]`)
        - **Popularity** (column ID: `[14]`)
        
        <br>
    
    3) **Clean string data** (remove extra spaces) with `str_clean` function. **No casing standardization will be perfromed** to avoid losing original casing information within Artist name. The `lower()` method will be applied temporarily without altering the value permanently and the original casing will be restored after the data cleaning. The `str_clean` function will be updated to support this.
    
    4) **Convert missing values into `None`** with `turn_none` function.
    
    5) **Convert numeric data into `integer` type** with `turn_int` function.
    
    6) **Convert out-of-bounds Popularity values into `None`** with `turn_none_pop` function.
    
    7) **Create a pre-final (pre-aggregation) vesrion dataset of dictionaris** of the following structure:
        ```
              {
               artist name: {
                        Top genre list: []
                        BPM list: []
                        Popularity list: []
                       }
              }
       
        ```

##### String Clean Function (Updated)

In [46]:
def str_clean(value):
    if value and type(value) == str:
        value = value.strip()  # .lower() and .title() methods removed
    return value

##### Row-by-Row Cleaning Process (Reinitiated)

In [47]:
dic_prefinal = {}

for i, row in enumerate(rows):
    
    if i in drop_row_id_list:
        continue
    if row[0] in missing_artist_row_id:
        continue

    artist = turn_none(str_clean(row[2]))
    genre = turn_none(str_clean(row[3]))
    bpm = turn_int(turn_none(str_clean(row[5])))
    pop = turn_none_pop(turn_int(turn_none(str_clean(row[14]))))

    if artist:
        dic_prefinal[artist] = dic_prefinal.get(artist, {
            "Top genre list": [],
            "BPM list": [],
            "Popularity list": []        
        })

        dic_prefinal[artist]["Top genre list"].append(genre)
        dic_prefinal[artist]["BPM list"].append(bpm)
        dic_prefinal[artist]["Popularity list"].append(pop)

for i, (key, val) in enumerate(dic_prefinal.items()):
    print(f"{i+1} {key}:")
    for k, v in val.items():
        print(f"   |{k}: {v}")

1 Norah Jones:
   |Top genre list: ['adult standards', 'adult standards']
   |BPM list: [157, 88]
   |Popularity list: [None, 74]
2 Deep Purple:
   |Top genre list: ['album rock', 'album rock', 'album rock', 'album rock']
   |BPM list: [135, 127, 114, 127]
   |Popularity list: [39, 45, 64, 37]
3 Gorillaz:
   |Top genre list: ['alternative hip hop', 'alternative hip hop']
   |BPM list: [168, 139]
   |Popularity list: [69, 80]
4 Foo Fighters:
   |Top genre list: ['alternative metal', 'alternative metal', 'alternative metal', 'alternative metal', 'alternative metal', 'alternative metal', 'alternative metal', 'alternative metal', 'alternative metal']
   |BPM list: [173, 168, 145, 130, 144, 138, 135, 158, 136]
   |Popularity list: [76, 72, 65, 75, 69, 69, 64, 77, None]
5 Bruce Springsteen:
   |Top genre list: ['classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classi

##### **CONCLUSION:**
---
- The resulting dictionary ends up containing **712 artist entries**, which is **2 more than the previous result** (beofore the `lower()` and `title()` methods removal from the cleaning process). This **might indicate that some duplicated entris might have been allowed** to the dictionary due to the inconsistent casing. **An addditional step, deduplication, will have to be performed** before continuing.
---

#### 4) --DUPLICATES IDENTIFICATION--

- The pre-final **dictionary keys (artist names) will be stored in a list**.

- **The `lower()` method** will be applied while iterating through the list values and **their instances counted** to identify duplicates.

##### Converting Dictionary into List of Keys

In [48]:
artist_names = [name for name in dic_prefinal]

##### Value Transformation and Count

In [49]:
val_count = {}
for val in artist_names:
    val = val.lower()
    val_count[val] = val_count.get(val, 0) + 1

for v in val_count:
    if val_count[v] > 1:
        print(v, val_count[v])

coldplay 3


##### **CONCLUSIONS:**
---
- **1 duplicated entry** was identified: "coldplay".

- **Variants of the duplicated artist name casing will be displayed**.

- **The rows associated with the duplicated entries will be inspected** to identify if the rows are **unique or duplicated** as it was performed during the earlier deduplication procedures.
---

##### Data Retrieval + Casing Variants Display

In [50]:
title_list = []
coldplay_set = set()
count = 0

for row in rows:
    artist = row[2]
    title = row[1]
    try:
        if artist.strip().lower() == "coldplay":
            count += 1
            print(count, row)
            title_list.append(title)
            if artist.strip() not in coldplay_set:
                coldplay_set.add(artist.strip())
        else:
            continue
    except:
        continue

print()
print(coldplay_set)

1 ['17', 'Speed of Sound', 'ColdPlay', 'permanent wave', '2005', '123', '90', '52', '-7', '7', '36', 'null', '0', '6', '69']
2 ['21', 'Fix You', 'coldplay ', 'permanent wave', '2005', '138', '42', '21', '-9', '11', '12', '296', '16', '3', '81']
3 ['31', 'the scientist', 'coldplay ', 'permanent wave', '2002', '146', '44', '56', '-7', '11', '21', '310', '73', '2', '84']
4 ['137', 'Talk', 'ColdPlay', 'permanent wave', '2005', '120', '56', '41', '-11', '16', '16', '311', '1', '3', '63']
5 ['156', 'God Put a Smile upon Your Face', 'ColdPlay', 'permanent wave', '2002', '127', '56', '61', '-6', '4', '24', '297', '18', '3', '64']
6 ['201', 'Green Eyes', 'Coldplay', 'permanent wave', '2002', '130', '41', '61', '-9', '23', '23', '223', '54', '3', '63']
7 ['240', 'Yellow', 'Coldplay', 'permanent wave', '2000', '173', '66', '43', '-7', '23', '28', '267', '0', '3', '82']
8 ['254', 'Amsterdam', 'Coldplay', 'permanent wave', '2002', '73', '18', '26', '-10', '12', '11', '319', '84', '3', '57']
9 ['262

##### Artist-Title Duplicated Relationship Inspection

In [51]:
title_count = {}

for title in title_list:
    title_count[title] = title_count.get(title, 0) + 1

for title in title_count:
    if title_count[title] > 1:
        print(title_count)

##### **CONCLUSIONS:**
---
- **No duplicated Artist-Title relationships identified** among the rows with "coldplay" artist entry.

- **3 different casing** instances of the artist name identified: "Coldplay", "ColdPlay", "coldplay". The correct one being "Coldplay".

- **The incorrect casing variants of the artist name will be corrected** during the reinitiated pre-final dictionary creation to avoid artist entry duplication.

---

#### 5) --DATA CLEANING & TRANSFORMATION (UPDATED 2)--

- The original data subset will be parsed and cleaned again and **the pre-final dictionary of artist entries will be created anew**.

- The earlier revealed duplicated artist ("Coldplay") name's **improper casing will be corrected** before adding it to the dictionaory.

- **The divider mistake will be corrected.** A dedicated counter will be implimented during valid Popularity values parsing to be then used as a divider for the Populartiy average calculation.

- **The final data will be organized into a dictionary of dictionaris** of the following structure:
    ```
    
          {
           artist name: {
                    Top genre:
                    Avg BPM:
                    Avg popularity:
                   }
          }
    ```

- 

##### Row-by-Row Cleaning Process (Reinitiated)

In [52]:
dic_prefinal = {}

for i, row in enumerate(rows):
    
    if i in drop_row_id_list:
        continue
    if row[0] in missing_artist_row_id:
        continue

    artist = turn_none(str_clean(row[2]))
    genre = turn_none(str_clean(row[3]))
    bpm = turn_int(turn_none(str_clean(row[5])))
    pop = turn_none_pop(turn_int(turn_none(str_clean(row[14]))))

    if artist:
        if artist.lower() == "coldplay":
            artist = "Coldplay"  # Correct duplicated artist casing
            dic_prefinal[artist] = dic_prefinal.get(artist, {
            "Top genre list": [],
            "BPM list": [],
            "Popularity list": []        
            })
        else:
            dic_prefinal[artist] = dic_prefinal.get(artist, {
                "Top genre list": [],
                "BPM list": [],
                "Popularity list": []        
            })

        dic_prefinal[artist]["Top genre list"].append(genre)
        dic_prefinal[artist]["BPM list"].append(bpm)
        dic_prefinal[artist]["Popularity list"].append(pop)

for i, (key, val) in enumerate(dic_prefinal.items()):
    print(f"{i+1} {key}:")
    for k, v in val.items():
        print(f"   |{k}: {v}")

1 Norah Jones:
   |Top genre list: ['adult standards', 'adult standards']
   |BPM list: [157, 88]
   |Popularity list: [None, 74]
2 Deep Purple:
   |Top genre list: ['album rock', 'album rock', 'album rock', 'album rock']
   |BPM list: [135, 127, 114, 127]
   |Popularity list: [39, 45, 64, 37]
3 Gorillaz:
   |Top genre list: ['alternative hip hop', 'alternative hip hop']
   |BPM list: [168, 139]
   |Popularity list: [69, 80]
4 Foo Fighters:
   |Top genre list: ['alternative metal', 'alternative metal', 'alternative metal', 'alternative metal', 'alternative metal', 'alternative metal', 'alternative metal', 'alternative metal', 'alternative metal']
   |BPM list: [173, 168, 145, 130, 144, 138, 135, 158, 136]
   |Popularity list: [76, 72, 65, 75, 69, 69, 64, 77, None]
5 Bruce Springsteen:
   |Top genre list: ['classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classic rock', 'classi

##### Data Aggregation Process (Final Dataset Dictionary)

In [53]:
dic_final = {}

for artist, dic in dic_prefinal.items():

    top_genre = dic["Top genre list"][0]
    bpm_total = 0
    pop_total = 0
    pop_val_count = 0
    
    for bpm_val in dic["BPM list"]:
        bpm_total += bpm_val
    bmp_avg = round(bpm_total / len(dic["BPM list"]))

    for pop_val in dic["Popularity list"]:
        if pop_val:
            pop_val_count +=1  # Implement value count to be used as divider
            pop_total += pop_val
    if pop_total > 0:
        pop_avg = round(pop_total / pop_val_count)  # Value count as divider
    
    dic_final[artist] = artist
    
    dic_final[artist] = {
        "Top genre": top_genre,
        "Avg BPM": bmp_avg,
        "Avg popularity": pop_avg
    }

for i, (artist, dic) in enumerate(dic_final.items()):
    print(f"{i + 1} {artist}:")
    for key, val in dic.items():
        print(f"   |{key}: {val}")
        

1 Norah Jones:
   |Top genre: adult standards
   |Avg BPM: 122
   |Avg popularity: 74
2 Deep Purple:
   |Top genre: album rock
   |Avg BPM: 126
   |Avg popularity: 46
3 Gorillaz:
   |Top genre: alternative hip hop
   |Avg BPM: 154
   |Avg popularity: 74
4 Foo Fighters:
   |Top genre: alternative metal
   |Avg BPM: 147
   |Avg popularity: 71
5 Bruce Springsteen:
   |Top genre: classic rock
   |Avg BPM: 120
   |Avg popularity: 59
6 City To City:
   |Top genre: alternative pop rock
   |Avg BPM: 99
   |Avg popularity: 45
7 Maroon 5:
   |Top genre: pop
   |Avg BPM: 102
   |Avg popularity: 74
8 Muse:
   |Top genre: modern rock
   |Avg BPM: 130
   |Avg popularity: 63
9 The Killers:
   |Top genre: modern rock
   |Avg BPM: 140
   |Avg popularity: 73
10 Eminem:
   |Top genre: detroit hip hop
   |Avg BPM: 116
   |Avg popularity: 76
11 Elvis Presley:
   |Top genre: adult standards
   |Avg BPM: 120
   |Avg popularity: 59
12 The White Stripes:
   |Top genre: alternative rock
   |Avg BPM: 124
   |Avg

##### **CONCLUSIONS:**
---
- **710 Artist entries have been added** to the pre-final dictionary after the data cleaning code corrections, which matches the earlier result.

- The mistake within Popularity average calculation was corrected through implementation of a dedicated value counter instead of list length (which allowed `None` Popularity values into the calculation).

- **The aggregation process perfromed on 710 Artist entries**.

- Next, **data validation checks** will be performed again.

---

#### 6) --VALIDATION CHECKS--

- **A number of validation chekcs will be performed again** on the resulting dictionary before the final dataset export.

- **The same Artist entries will be selected** for validation as during the earlier validation stage.

- The selected entries data will be retreived from the original dataset, their data inspected and the average numbers recalculated, to be then compared with the corresponding entry within the final dictionary.

#### --Selective Data Validation Check #1: "Muse"

In [54]:
value = "Muse"
print(value)
print()

count = 0
bpm_count = 0
pop_count = 0
genres = []
bpm_total = 0
pop_total = 0

for row in rows:
    artist = row[2]
    genre = row[3]
    bpm = row[5]
    pop = row[14]
    
    if artist.strip().lower() == value.lower():
        count += 1
        genres.append(genre)
        if bpm and bpm.isdigit():
            bpm_count += 1
            bpm = int(bpm)
            bpm_total += bpm
        if pop and pop.isdigit():
            pop_count += 1
            pop = int(pop)
            pop_total += pop
        
        print(row)

bpm_avg = round(bpm_total / bpm_count)
pop_avg = round(pop_total / pop_count)

print()
print("| Featured genres:", genres)
print("| Average BPM:", bpm_avg)
print("| Aberage popularity:", pop_avg)

if value in dic_final:
    print()
    print("Dictionary record found:", value)
    print(dic_final[value])
else:
    print()
    print(f"<< Dictionary record not found: {value} >>")


Muse

['8', 'Knights of Cydonia', 'Muse', 'modern rock', '2006', '137', '96', '37', '-5', '12', '21', '366', '0', '14', '69']
['117', 'Feeling Good', 'Muse', 'modern rock', '2001', '109', '42', '35', '-8', '10', '27', '199', '30', '3', '51']
['165', 'Uprising', 'Muse', 'modern rock', '2009', '128', '91', '60', '-4', '12', '41', '305', '0', '8', '76']
['189', 'Plug in Baby', 'Muse', 'modern rock', '2001', '136', '97', '41', '-4', '11', '35', '218', '0', '5', '53']
['198', 'Starlight', 'Muse', 'modern rock', '2006', '122', '87', '55', '-4', '21', '32', '240', '0', '3', '73']
['204', 'Supermassive Black Hole', 'Muse', 'modern rock', '2006', '120', '92', '67', '-4', '9', '78', '212', '5', '4', '73']
['255', 'Time Is Running Out', 'Muse', 'modern rock', '2004', '118', '84', '59', '-6', '9', '43', '237', '0', '6', '69']
['315', '  Sing for Absolution  ', 'Muse', 'modern rock', '2004', '170', '68', '44', '-7', '9', '19', '295', '46', '3', '57']
['339', 'Resistance', 'Muse', 'modern rock', '20

#### --Selective Data Validation Check #2: "AC/DC"

In [55]:
value = "AC/DC"
print(value)
print()

count = 0
bpm_count = 0
pop_count = 0
genres = []
bpm_total = 0
pop_total = 0

for row in rows:
    artist = row[2]
    genre = row[3]
    bpm = row[5]
    pop = row[14]
    
    if artist.strip().lower() == value.lower():
        count += 1
        genres.append(genre)
        if bpm and bpm.isdigit():
            bpm_count += 1
            bpm = int(bpm)
            bpm_total += bpm
        if pop and pop.isdigit():
            pop_count += 1
            pop = int(pop)
            pop_total += pop
        
        print(row)

bpm_avg = round(bpm_total / bpm_count)
pop_avg = round(pop_total / pop_count)

print()
print("| Featured genres:", genres)
print("| Average BPM:", bpm_avg)
print("| Aberage popularity:", pop_avg)

if value in dic_final:
    print()
    print("Dictionary record found:", value)
    print(dic_final[value])
else:
    print()
    print(f"<< Dictionary record not found: {value} >>")


AC/DC

['1036', 'Whole Lotta Rosie', 'AC/DC', 'album rock', '1977', '159', '84', '29', '-4', '10', '42', '334', '0', '8', '65']
['1054', 'Let There Be Rock', 'AC/DC', 'album rock', '1977', '91', '73', '49', '-6', '8', '43', '366', '0', '9', '']
['1114', 'Highway to Hell', 'AC/DC', 'album rock', '1979', '116', '91', '57', '-5', '16', '42', '', '6', '13', '83']
['1158', 'Back In Black', 'AC/DC', 'album rock', '1980', '188', '70', '31', '-6', '8', '76', '', '1', '5', '83']
['1161', 'You Shook Me All Night Long', 'AC/DC', 'album rock', '1980', '127', '77', '53', '-6', '39', '76', '210', '0', '6', '']
['1497', 'Thunderstruck', 'AC/DC', 'album rock', '1990', '134', '89', '50', '-5', '22', '26', '293', '0', '4', '81']

| Featured genres: ['album rock', 'album rock', 'album rock', 'album rock', 'album rock', 'album rock']
| Average BPM: 136
| Aberage popularity: 78

Dictionary record found: AC/DC
{'Top genre': 'album rock', 'Avg BPM': 136, 'Avg popularity': 78}


#### --Selective Data Validation Check #3: "OneRepublic"

In [56]:
value = "OneRepublic"
print(value)
print()

count = 0
bpm_count = 0
pop_count = 0
genres = []
bpm_total = 0
pop_total = 0

for row in rows:
    artist = row[2]
    genre = row[3]
    bpm = row[5]
    pop = row[14]
    
    if artist.strip().lower() == value.lower():
        count += 1
        genres.append(genre)
        if bpm and bpm.isdigit():
            bpm_count += 1
            bpm = int(bpm)
            bpm_total += bpm
        if pop and pop.isdigit():
            pop_count += 1
            pop = int(pop)
            pop_total += pop
        
        print(row)

bpm_avg = round(bpm_total / bpm_count)
pop_avg = round(pop_total / pop_count)

print()
print("| Featured genres:", genres)
print("| Average BPM:", bpm_avg)
print("| Aberage popularity:", pop_avg)

if value in dic_final:
    print()
    print("Dictionary record found:", value)
    print(dic_final[value])
else:
    print()
    print(f"<< Dictionary record not found: {value} >>")


OneRepublic

['596', 'Counting Stars', 'OneRepublic', 'dance pop', '2014', '122', '71', '66', '-5', '12', '48', '258', '7', '4', '74']

| Featured genres: ['dance pop']
| Average BPM: 122
| Aberage popularity: 74

Dictionary record found: OneRepublic
{'Top genre': 'dance pop', 'Avg BPM': 122, 'Avg popularity': 74}


#### --Selective Data Validation Check #4: "Coldplay"

In [57]:
value = "Coldplay"
print(value)
print()

count = 0
bpm_count = 0
pop_count = 0
genres = []
bpm_total = 0
pop_total = 0

for row in rows:
    artist = row[2]
    genre = row[3]
    bpm = row[5]
    pop = row[14]
    
    if artist.strip().lower() == value.lower():
        count += 1
        genres.append(genre)
        if bpm and bpm.isdigit():
            bpm_count += 1
            bpm = int(bpm)
            bpm_total += bpm
        if pop and pop.isdigit():
            pop_count += 1
            pop = int(pop)
            pop_total += pop
        
        print(row)

bpm_avg = round(bpm_total / bpm_count)
pop_avg = round(pop_total / pop_count)

print()
print("| Featured genres:", genres)
print("| Average BPM:", bpm_avg)
print("| Aberage popularity:", pop_avg)

if value in dic_final:
    print()
    print("Dictionary record found:", value)
    print(dic_final[value])
else:
    print()
    print(f"<< Dictionary record not found: {value} >>")


Coldplay

['17', 'Speed of Sound', 'ColdPlay', 'permanent wave', '2005', '123', '90', '52', '-7', '7', '36', 'null', '0', '6', '69']
['21', 'Fix You', 'coldplay ', 'permanent wave', '2005', '138', '42', '21', '-9', '11', '12', '296', '16', '3', '81']
['31', 'the scientist', 'coldplay ', 'permanent wave', '2002', '146', '44', '56', '-7', '11', '21', '310', '73', '2', '84']
['137', 'Talk', 'ColdPlay', 'permanent wave', '2005', '120', '56', '41', '-11', '16', '16', '311', '1', '3', '63']
['156', 'God Put a Smile upon Your Face', 'ColdPlay', 'permanent wave', '2002', '127', '56', '61', '-6', '4', '24', '297', '18', '3', '64']
['201', 'Green Eyes', 'Coldplay', 'permanent wave', '2002', '130', '41', '61', '-9', '23', '23', '223', '54', '3', '63']
['240', 'Yellow', 'Coldplay', 'permanent wave', '2000', '173', '66', '43', '-7', '23', '28', '267', '0', '3', '82']
['254', 'Amsterdam', 'Coldplay', 'permanent wave', '2002', '73', '18', '26', '-10', '12', '11', '319', '84', '3', '57']
['262', 'In M

##### **CONCLUSIONS:**
---

- 4 validation checks were performed.

- An arbitrary Artist entry was selected and retreived from the original dataset**, its data inspected and the average numbers recalculated, to be then compared with the corresponding entry within the final dictionary.

- Check #1 and #2 are identical to the validation checks that have been performed earlier. During the earliear data validation process, check #2 (featuring "AC/DC" value) failed due to the careless approach toward string data manipularion and the loss of signature casing information.

- After the necessary changes have been done and critical mistakes corrected, **validation check #2 succeeds**.

- Finally, **all 4 validation checks indicate consistency** between the data retrieved from the original dataset and the corresponding one retreived from the final dictionary. Artist names, top genres, average BPM numbers and average popularity numbers match which **can be evidence of data integrity preservation**, also implying that the data preparation process was held successfully.

- **The final dictinary therefore will be exported into a csv file as the project's final dataset.**

---

### STAGE 6 - DATA EXPORT

- The final dictionary **will be exported into a JSON file with `json` library** to preserve the hierarchical structure of the final data.

##### Dictionary Export

In [58]:
import json

with open("Spotify-2000-Artist_Page.json", "w", encoding="utf-8") as outfile:
    json.dump(dic_final, outfile, indent=4, ensure_ascii=False)

## 4. PROJECT SUMMARY

- This project was an early attempt to work through a complete data-preparation process using music data as the domain context.

- Starting from a synthetically corrupted dataset, I defined a concrete data objective, inspected the available information, identified and investigated data-quality issues, applied cleaning and transformation procedures, and validated the resulting data before export.

- A particular focus was placed on **preserving information where possible rather than removing or modifying data without sufficient justification**. This required distinguishing genuine duplicates from candidate duplicates, handling missing and out-of-bounds values according to their role in the intended output, and considering domain-specific characteristics such as BPM interpretation.

- The final pipeline produces a simplified Artist Page lookup structure containing aggregated artist-level information such as average BPM, average popularity, and most frequent genre.

- The project also demonstrated the value of iterative inspection and validation. Several issues in the initial approaches were revealed only through repeated review, correction, and comparison with the source data. This made the project useful not only as an implementation exercise, but also as practical experience in reasoning about data quality and transformation decisions.


---

## 5. CHALLENGES & LESSONS

- One of the main lessons from the project was that the difficulty of a data-preparation task does not necessarily come from writing the code, highligting **the distinction between knowing a programming tool and knowing how to apply it effectively to a data problem.** That said, deeper and broader knowledge of the instrument itself allows to improve optimization and code architecture, reducing the number of steps required to achieve the same result. Overall, gaining more experience in data-preparation and domain-specific considerations should help in both directions and generally allow similar processes to become simpler and more efficient over time.

- One of the biggest challenges was **preventing the loss of valuable information during the data cleaning stage.** The loss of signature artist name casing was demonstrative in how **a technically correct transformation can produce undesirable results**, and how seemingly simple string operations such as `strip()` and `lower()` can lead to critical consequences. **The proper and well thought-out initial implementation of those methods could have helped avoid a number of mistakes** that took place during the data inspection and cleaning stages of this project, and also potentially reveal even more issues which could have been unintentionally allowed into the resulting final dataset, totally overlooked.

- In several cases, **the difficult part was deciding what the data represented and determining what should happen to it**. That was the case during the Artics column inspection, where, for example, one of the "Nan" values could legitimately be an established Artist name despite being a common sentinel value. In regard of such cases, **a dicitonary of exceptions could be created** listing all the sentinel-like artist names, which could be used for quick reference in future pipelines.

- The project demonstrated how domain knowledge influences data-cleaning decisions. **The intended meaning of the data must be understood when deciding whether a value is incorrect, unusual, missing, or simply represented differently.** E.g. values that appear unusual from a purely numerical perspective are not necessarily incorrect (as was the case with one outlier value revealed during the BPM inspection). 

- Expectedly, **data validation checks proved to be particularly useful** and helpful at finding data integrity loss cases or mistakes in calculations due to the incorrect code. The project also reinforced the importance of iterative inspection and validation and that these procedures should be part of the data-preparation process rather than a final afterthought. However, more kinds of validation procedures could have probably been designed and implemented for a more vigorous integrity inspection. Thus, honing validation skills and broadening validation methods toolkit is critical.

- The initial deduplication process became relatively laborious due to requiring several stages of identification, comparison, validation, and decision-making. The implemented code, though, may now be used as a foundation for further data cleaning projects.

- The documentation process itself initially slowed development, but it is what provided the project with a solid structure from the start and made the overall thinking process more coherent, focused, and accurate, which, in its turn, made debugging and reviewing previous approaches substantially easier.

- Overall, the project **provided a highly valuable experience with a complete data-preparation cycle**, **exposing problems that would be difficult to encounter through isolated exercises alone**. It revealed unexpected pitfalls which enforced active, complex and creative problem-solving. The project additionally provides possibilities for meta-cognitive personal analysis and can also serve as a point for future reference.

---

## 6. POLISHED VERSION

- A shortened and more structured presentation of the resulting approach is available here: [Music Data Preparation Pipeline - Polished Version](https://github.com/ed-cybros/Music-Data-Preparation-Pipeline-Python-Only---Polished-Version).